<a href="https://colab.research.google.com/github/muratal49/DroneStabilization_using_GNNs_DRL/blob/main/drone_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SECOND MODEL: **

In [1]:
#import libraries:
%cd /content/drive/MyDrive/Drone\ Stabilization\ GNN+Deep\ RL
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
# MODELS:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader, Dataset
import torch.nn.functional as F

/content/drive/MyDrive/Drone Stabilization GNN+Deep RL


In [16]:
df=pd.read_csv('full_df.csv')

In [17]:
df=df.drop(columns=['speed'])
df

,wind_speed,wind_angle,battery_voltage,battery_current,position_z,orientation_x,orientation_y,orientation_z,orientation_w,velocity_x,...,angular_y,angular_z,linear_acceleration_x,linear_acceleration_y,linear_acceleration_z,payload,roll,pitch,yaw,throttle_proxy
0,3.2,342.0,21.339952,21.986654,246.390305,0.006542,-0.024580,0.249651,0.968002,0.040231,...,0.032387,0.143255,-0.136207,0.475946,-9.810058,0.0,0.000394,-0.050876,0.504796,0.000000
1,3.6,328.0,21.400051,23.171469,246.679753,0.015266,-0.030405,0.262511,0.964329,-0.052576,...,0.156187,-0.139474,0.081773,0.514394,-9.337005,0.0,0.013510,-0.066704,0.531112,0.021429
2,1.6,256.0,22.200111,21.318707,253.855834,0.031600,0.018123,0.286017,0.957532,-0.065122,...,-0.122464,0.090408,0.218135,0.263926,-9.688867,0.0,0.070953,0.016631,0.581122,0.005488
3,3.6,36.0,21.656721,19.116064,246.044628,-0.006235,0.000546,0.257715,0.966201,0.080457,...,0.070794,-0.049493,-0.170963,-0.072780,-9.623001,0.0,-0.011767,0.004268,0.521299,0.008472
4,1.3,140.0,22.225153,22.598942,249.060464,0.015383,0.025392,0.072014,0.996962,-0.020177,...,-0.104467,0.032496,0.299177,-0.014806,-9.957019,0.0,0.034377,0.048432,0.145049,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,1.1,192.0,22.230160,22.026413,285.728395,0.006177,-0.004996,-0.615852,0.787822,0.013478,...,0.082361,-0.004988,0.274732,0.141522,-9.623850,250.0,0.015886,-0.000264,-1.326985,0.008434
69996,2.9,340.0,21.113331,22.694363,312.390062,-0.029730,-0.041595,-0.707873,0.704487,-0.385310,...,-0.143677,0.085098,0.170801,0.352779,-10.052003,0.0,0.017087,-0.100868,-1.576453,0.000000
69997,1.8,123.0,21.545288,21.756054,273.163826,0.020698,0.004090,0.992034,0.124190,-0.060702,...,-0.042420,-0.139079,0.113167,0.555813,-9.759626,0.0,0.013266,-0.040062,2.892248,0.002282
69998,0.8,133.0,23.633709,-0.015904,270.750332,-0.013848,0.001241,-0.971395,0.237063,0.012214,...,-0.001997,-0.009251,-0.083768,0.279890,-9.804604,250.0,-0.008980,-0.026319,-2.662744,0.000244


## BC_2

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.transform import Rotation as R

# Data preparation
def prepare_bc_data(df):
    """Prepare data for BC training"""

    # Compute Euler angles and throttle proxy
    r = R.from_quat(df[['orientation_x', 'orientation_y', 'orientation_z', 'orientation_w']].values)
    euler = r.as_euler('xyz', degrees=False)
    df['roll'], df['pitch'], df['yaw'] = euler[:, 0], euler[:, 1], euler[:, 2]

    # Compute throttle proxy
    g = 9.81
    df['throttle_proxy'] = df['linear_acceleration_z'] + g

    # Select input features (excluding speed, but keeping wind and payload)
    input_features = [
        'wind_speed', 'wind_angle', 'payload',  # From metadata
        'battery_voltage', 'battery_current',
         'position_z',
        'velocity_x', 'velocity_y', 'velocity_z',
        'linear_acceleration_x', 'linear_acceleration_y', 'linear_acceleration_z',
        'roll', 'pitch', 'yaw'  # Derived from quaternions
    ]

    # Target features (angular rates + throttle proxy)
    target_features = ['angular_x', 'angular_y', 'angular_z', 'throttle_proxy']

    X = df[input_features].values
    y = df[target_features].values

    return X, y, input_features, target_features

# Dataset class
class DroneDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# BC Model
class BCModel(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims=[256, 128, 64]):
        super(BCModel, self).__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Training function
def train_bc_model(X_train, y_train, X_val, y_val, epochs=100, batch_size=64, lr=0.001):

    # Create datasets and dataloaders
    train_dataset = DroneDataset(X_train, y_train)
    val_dataset = DroneDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    input_dim = X_train.shape[1]
    output_dim = y_train.shape[1]  # Should be 4: [angular_x, angular_y, angular_z, throttle_proxy]

    model = BCModel(input_dim, output_dim)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

    # Training loop
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        if epoch % 10 == 0:
            print(f'Epoch {epoch}: Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')

    return model, train_losses, val_losses

# Main training script
def main():
    # Load your data
    # df = pd.read_csv('your_flight_data.csv')  # Replace with your data file

    # Prepare data
    X, y, input_features, target_features = prepare_bc_data(df)

    print(f"Input features ({len(input_features)}): {input_features}")
    print(f"Target features ({len(target_features)}): {target_features}")
    print(f"Data shape: X={X.shape}, y={y.shape}")

    # Scale features
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)

    # Split data (you might want to split by flight to avoid data leakage)
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )

    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")

    # Train model
    model, train_losses, val_losses = train_bc_model(
        X_train, y_train, X_val, y_val,
        epochs=200, batch_size=64, lr=0.001
    )

    # Save model and scalers
    torch.save(model.state_dict(), 'bc_stabilization_model.pth')
    torch.save(scaler_X, 'scaler_X.pth')
    torch.save(scaler_y, 'scaler_y.pth')

    print("Training completed! Model and scalers saved.")

    return model, scaler_X, scaler_y

# if __name__ == "__main__":
#     model, scaler_X, scaler_y = main()

In [19]:
#data  preparation
X, y, input_features, target_features = prepare_bc_data(df)
#Scaling:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

#split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

#trainig BC2:
model, train_losses, val_losses = train_bc_model(
    X_train, y_train, X_val, y_val,
    epochs=200, batch_size=64, lr=0.001
)

Epoch 0: Train Loss: 0.694350, Val Loss: 0.629094
Epoch 10: Train Loss: 0.486994, Val Loss: 0.495371
Epoch 20: Train Loss: 0.455675, Val Loss: 0.469123
Epoch 30: Train Loss: 0.437342, Val Loss: 0.463095
Epoch 40: Train Loss: 0.423545, Val Loss: 0.458741
Epoch 50: Train Loss: 0.417033, Val Loss: 0.445433
Epoch 60: Train Loss: 0.410950, Val Loss: 0.445654
Epoch 70: Train Loss: 0.390215, Val Loss: 0.435695
Epoch 80: Train Loss: 0.380216, Val Loss: 0.435601
Epoch 90: Train Loss: 0.380761, Val Loss: 0.433960
Epoch 100: Train Loss: 0.372714, Val Loss: 0.432073
Epoch 110: Train Loss: 0.369725, Val Loss: 0.431287
Epoch 120: Train Loss: 0.365215, Val Loss: 0.429977
Epoch 130: Train Loss: 0.364746, Val Loss: 0.430987
Epoch 140: Train Loss: 0.363418, Val Loss: 0.430624
Epoch 150: Train Loss: 0.359309, Val Loss: 0.427285
Epoch 160: Train Loss: 0.357393, Val Loss: 0.429384
Epoch 170: Train Loss: 0.354768, Val Loss: 0.427867
Epoch 180: Train Loss: 0.357843, Val Loss: 0.428037
Epoch 190: Train Loss: 

## EXTREME2 real physics, 3 drone types

In [ ]:
pip install 'shimmy>=2.0'

In [ ]:
!pip install ipywidgets

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import os
import pybullet as p
from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ActionType, ObservationType, DroneModel
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback
import tempfile

class CustomTumblingHoverAviary(HoverAviary):
    """Custom HoverAviary with tumbling, wind, payload, and multiple drone designs"""

    def __init__(self,
                 initial_ang_vel_range=(-2, 2),
                 wind_speed_range=(0, 5),
                 wind_angle_range=(0, 2*np.pi),
                 payload_range=(0, 0.2),
                 drone_design="default",
                 **kwargs):

        self.initial_ang_vel_range = initial_ang_vel_range
        self.wind_speed_range = wind_speed_range
        self.wind_angle_range = wind_angle_range
        self.payload_range = payload_range
        self.drone_design = drone_design

        # Use default drone model for now (custom URDF creation is complex)
        super().__init__(**kwargs)

        # Store original mass for payload calculations
        self.original_mass = self.M

        # Apply drone design modifications after initialization
        self._apply_drone_design_modifications()

    def _apply_drone_design_modifications(self):
        """Apply design-specific modifications to the drone"""

        # Drone specifications
        design_multipliers = {
            "default": {"mass_mult": 1.0, "thrust_mult": 1.0},
            "medium": {"mass_mult": 18.5, "thrust_mult": 20.0},  # 500g vs 27g
            "large": {"mass_mult": 74.0, "thrust_mult": 80.0}    # 2kg vs 27g
        }

        if self.drone_design in design_multipliers:
            mult = design_multipliers[self.drone_design]

            # Modify mass
            self.M = self.original_mass * mult["mass_mult"]

            # Modify thrust coefficients
            self.KF = self.KF * mult["thrust_mult"]
            self.KM = self.KM * mult["thrust_mult"]

            # Update PyBullet mass
            for i in range(self.NUM_DRONES):
                p.changeDynamics(
                    self.DRONE_IDS[i],
                    -1,  # base link
                    mass=self.M,
                    physicsClientId=self.CLIENT
                )

    def reset(self, seed=None, options=None):
        """Reset with tumbling, wind, and payload"""
        obs, info = super().reset(seed=seed, options=options)

        # Apply random initial angular velocity (tumbling)
        if abs(self.initial_ang_vel_range[1]) > 0:
            initial_ang_vel = np.random.uniform(
                self.initial_ang_vel_range[0],
                self.initial_ang_vel_range[1],
                size=3
            )

            # Set angular velocity in PyBullet
            for i in range(self.NUM_DRONES):
                p.resetBaseVelocity(
                    self.DRONE_IDS[i],
                    linearVelocity=[0, 0, 0],
                    angularVelocity=initial_ang_vel,
                    physicsClientId=self.CLIENT
                )

        # Set random wind parameters
        self.current_wind_speed = np.random.uniform(*self.wind_speed_range)
        self.current_wind_angle = np.random.uniform(*self.wind_angle_range)

        # Set random payload (modify mass)
        self.current_payload = np.random.uniform(*self.payload_range)
        new_mass = self.M + self.current_payload  # M already includes design modifications

        for i in range(self.NUM_DRONES):
            p.changeDynamics(
                self.DRONE_IDS[i],
                -1,  # base link
                mass=new_mass,
                physicsClientId=self.CLIENT
            )

        return obs, info

    def step(self, action):
        """Step with wind forces applied"""
        # Apply wind forces before physics step
        if hasattr(self, 'current_wind_speed') and self.current_wind_speed > 0:
            # Scale wind force based on drone size
            wind_scale = 0.01 if self.drone_design == "default" else (0.02 if self.drone_design == "medium" else 0.03)
            wind_force_x = self.current_wind_speed * np.cos(self.current_wind_angle) * wind_scale
            wind_force_y = self.current_wind_speed * np.sin(self.current_wind_angle) * wind_scale

            for i in range(self.NUM_DRONES):
                p.applyExternalForce(
                    self.DRONE_IDS[i],
                    -1,  # base link
                    [wind_force_x, wind_force_y, 0],
                    [0, 0, 0],  # world coordinates
                    p.WORLD_FRAME,
                    physicsClientId=self.CLIENT
                )

        obs, reward, terminated, truncated, info = super().step(action)

        # Override reward with custom stability + height + survival reward
        reward = self._computeCustomReward()

        return obs, reward, terminated, truncated, info

    def _computeCustomReward(self):
        """Custom reward: stability + altitude + survival"""
        state = self._getDroneStateVector(0)

        # Angular velocity (stability)
        ang_vel = state[13:16]  # p, q, r
        ang_vel_norm = np.linalg.norm(ang_vel)

        # Position
        z_pos = state[2]

        # Rewards
        alive_bonus = 1.0
        stability_reward = np.exp(-0.5 * ang_vel_norm)
        altitude_reward = np.exp(-abs(z_pos - 1.0))  # target 1m altitude

        # Time bonus for faster stabilization
        time_bonus = max(0, (self.EPISODE_LEN_SEC * self.PYB_FREQ - self.step_counter) / (self.EPISODE_LEN_SEC * self.PYB_FREQ)) * 0.5

        total_reward = alive_bonus + stability_reward + altitude_reward + time_bonus
        return total_reward

    def _computeTruncated(self):
        """Custom termination conditions"""
        state = self._getDroneStateVector(0)

        # Position bounds
        if (abs(state[0]) > 5.0 or abs(state[1]) > 5.0 or
            state[2] < 0.1 or state[2] > 10.0):
            return True

        # Orientation bounds (prevent extreme flips)
        if abs(state[7]) > 0.9 or abs(state[8]) > 0.9:  # roll, pitch
            return True

        # Time limit
        if self.step_counter / self.PYB_FREQ > self.EPISODE_LEN_SEC:
            return True

        return False

# Updated curriculum phases with 500K final phase
curriculum_phases = [
    {
        "name": "Phase1_Basic",
        "wind_speed_range": (0, 5),
        "payload_range": (0, 0.2),
        "ang_vel_range": (-1, 1),
        "timesteps": 200_000
    },
    {
        "name": "Phase2_Moderate",
        "wind_speed_range": (0, 10),
        "payload_range": (0, 0.5),
        "ang_vel_range": (-3, 3),
        "timesteps": 300_000
    },
    {
        "name": "Phase3_Tumbling",
        "wind_speed_range": (0, 20),
        "payload_range": (0, 0.8),
        "ang_vel_range": (-4, 4),
        "timesteps": 400_000
    },
    {
        "name": "Phase4_Extreme",
        "wind_speed_range": (0, 30),
        "payload_range": (0, 1.0),
        "ang_vel_range": (-6, 6),
        "timesteps": 500_000  # Reduced to 500K
    }
]

def make_env(phase_config, drone_design="default"):
    def _init():
        env = CustomTumblingHoverAviary(
            initial_ang_vel_range=phase_config["ang_vel_range"],
            wind_speed_range=phase_config["wind_speed_range"],
            payload_range=phase_config["payload_range"],
            drone_design=drone_design,
            act=ActionType.PID,
            obs=ObservationType.KIN,
            gui=False,
            record=False
        )
        return env
    return _init

# Create directories with July15 prefix
os.makedirs('./July15_models', exist_ok=True)
os.makedirs('./July15_logs', exist_ok=True)

# RESUME DEFAULT TRAINING FROM MODERATE 200K CHECKPOINT
print(f"\n{'='*60}")
print(f"RESUMING DEFAULT DRONE TRAINING FROM MODERATE 200K CHECKPOINT")
print(f"{'='*60}")

# Load the checkpoint
checkpoint_path = "./July15_models/July15_default_check_Phase2_Moderate/July15_model_default_Phase2_Moderate_200000_steps.zip"
print(f"🔄 Loading checkpoint: {checkpoint_path}")

# Create environment for Phase3 (Tumbling)
phase3 = curriculum_phases[2]  # Phase3_Tumbling
env = DummyVecEnv([make_env(phase3, "default") for _ in range(4)])

# Load model from checkpoint
model = PPO.load(checkpoint_path, env=env, verbose=1)
model.tensorboard_log = f"./July15_logs/July15_log_default/"

print(f"\n🚀 Continuing with Phase3_Tumbling for default drone:")
print(f"  Wind: {phase3['wind_speed_range']} m/s")
print(f"  Payload: {phase3['payload_range']} kg")
print(f"  Angular velocity: {phase3['ang_vel_range']} rad/s")
print(f"  Timesteps: {phase3['timesteps']:,}")

# Create checkpoint directory for Phase3
checkpoint_dir = f"./July15_models/July15_default_check_{phase3['name']}/"
os.makedirs(checkpoint_dir, exist_ok=True)

# Checkpoint callback for Phase3
checkpoint_callback = CheckpointCallback(
    save_freq=25000,
    save_path=checkpoint_dir,
    name_prefix=f"July15_model_default_{phase3['name']}"
)

# Train Phase3
model.learn(
    total_timesteps=phase3["timesteps"],
    callback=checkpoint_callback,
    progress_bar=True
)

# Save Phase3 final model
model_path = f"./July15_models/July15_model_default_{phase3['name']}_final"
model.save(model_path)
print(f"✅ Saved Phase3 final model: {model_path}.zip")

env.close()

# Continue with Phase4 (Extreme) for default
print(f"\n🚀 Continuing with Phase4_Extreme for default drone:")
phase4 = curriculum_phases[3]  # Phase4_Extreme
env = DummyVecEnv([make_env(phase4, "default") for _ in range(4)])
model.set_env(env)

print(f"  Wind: {phase4['wind_speed_range']} m/s")
print(f"  Payload: {phase4['payload_range']} kg")
print(f"  Angular velocity: {phase4['ang_vel_range']} rad/s")
print(f"  Timesteps: {phase4['timesteps']:,}")

# Create checkpoint directory for Phase4
checkpoint_dir = f"./July15_models/July15_default_check_{phase4['name']}/"
os.makedirs(checkpoint_dir, exist_ok=True)

# Checkpoint callback for Phase4
checkpoint_callback = CheckpointCallback(
    save_freq=25000,
    save_path=checkpoint_dir,
    name_prefix=f"July15_model_default_{phase4['name']}"
)

# Train Phase4
model.learn(
    total_timesteps=phase4["timesteps"],
    callback=checkpoint_callback,
    progress_bar=True
)

# Save Phase4 final model
model_path = f"./July15_models/July15_model_default_{phase4['name']}_final"
model.save(model_path)
print(f"✅ Saved Phase4 final model: {model_path}.zip")

env.close()

print(f"✅ DEFAULT DRONE TRAINING COMPLETE!")

# NOW TRAIN MEDIUM AND LARGE FROM SCRATCH (with 500K final phase)
drone_designs = ["medium", "large"]

for drone_design in drone_designs:
    print(f"\n{'='*60}")
    print(f"TRAINING DRONE DESIGN: {drone_design.upper()}")
    print(f"{'='*60}")

    model = None

    for i, phase in enumerate(curriculum_phases):
        print(f"\nStarting {phase['name']} for {drone_design} drone with config:")
        print(f"  Wind: {phase['wind_speed_range']} m/s")
        print(f"  Payload: {phase['payload_range']} kg")
        print(f"  Angular velocity: {phase['ang_vel_range']} rad/s")
        print(f"  Timesteps: {phase['timesteps']:,}")
        print(f"  Using 4 parallel environments")

        # Create 4 parallel environments
        env = DummyVecEnv([make_env(phase, drone_design) for _ in range(4)])

        # Create or update model
        if model is None:
            model = PPO(
                "MlpPolicy",
                env,
                verbose=1,
                batch_size=256,
                n_steps=2048,
                learning_rate=3e-4,
                tensorboard_log=f"./July15_logs/July15_log_{drone_design}/"
            )
        else:
            model.set_env(env)

        # Create checkpoint directory with July15 prefix
        checkpoint_dir = f"./July15_models/July15_{drone_design}_check_{phase['name']}/"
        os.makedirs(checkpoint_dir, exist_ok=True)

        # Add checkpoint callback - save every 50K steps
        checkpoint_callback = CheckpointCallback(
            save_freq=25000,
            save_path=checkpoint_dir,
            name_prefix=f"July15_model_{drone_design}_{phase['name']}"
        )

        # Train
        print(f"📁 Checkpoints will be saved to: {checkpoint_dir}")
        print(f"📊 TensorBoard logs: ./July15_logs/July15_log_{drone_design}/")

        model.learn(
            total_timesteps=phase["timesteps"],
            callback=checkpoint_callback,
            progress_bar=True
        )

        # Save final model with July15 prefix
        model_path = f"./July15_models/July15_model_{drone_design}_{phase['name']}_final"
        model.save(model_path)
        print(f"✅ Saved final model: {model_path}.zip")

        env.close()

print(f"\n{'='*60}")
print("🎉 ALL TRAINING COMPLETE!")
print(f"{'='*60}")

# Print final summary
print("\nTRAINING SUMMARY:")
print("="*50)
print("✅ DEFAULT: Resumed from Moderate 200K → Completed Tumbling + Extreme (500K)")
print("✅ MEDIUM: Full training from scratch → All phases (500K final)")
print("✅ LARGE: Full training from scratch → All phases (500K final)")

print("\nFINAL MODELS SAVED:")
print("="*30)
print("📦 July15_model_default_Phase4_Extreme_final.zip")
print("📦 July15_model_medium_Phase4_Extreme_final.zip")
print("📦 July15_model_large_Phase4_Extreme_final.zip")

   2% ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8,180/400,000  [ 0:00:34 < 0:27:02 , 242 it/s ]

## VISUALOZATION/DASHBOARD

In [ ]:
!nvidia-smi

In [4]:
 pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 41.5 MB/s eta 0:00:00


In [25]:
import os
import numpy as np
import matplotlib.pyplot as plt
import imageio
import time
import pybullet as p
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from IPython.display import Image as IPyImage, display as ipy_display
import ipywidgets as widgets
from ipywidgets import VBox, FloatSlider, FloatText, Button, Output

# --- Virtual display setup (if needed) ---
from pyvirtualdisplay import Display
display_instance = None
def start_virtual_display():
    global display_instance
    if display_instance is None:
        display_instance = Display(visible=0, size=(1024, 768))
        display_instance.start()
start_virtual_display()

# --- Your CurriculumHoverAviary environment ---
from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ActionType, ObservationType

class CurriculumHoverAviary(HoverAviary):
    def __init__(self,
                 wind_speed_range=(0, 2),
                 wind_angle_range=(0, 360),
                 payload_range=(0, 100),
                 tumbling_range=(0, 0),
                 **kwargs):

        self.wind_speed_range = wind_speed_range
        self.wind_angle_range = wind_angle_range
        self.payload_range = payload_range
        self.tumbling_range = tumbling_range

        super().__init__(**kwargs)

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)

        self.current_wind_speed = np.random.uniform(*self.wind_speed_range)
        self.current_wind_angle = np.random.uniform(*self.wind_angle_range)
        self.current_payload = np.random.uniform(*self.payload_range)
        if hasattr(self, 'PAYLOAD'):
            self.PAYLOAD = self.current_payload / 1000.0

        if self.tumbling_range[1] > 0:
            initial_angular_vel = np.random.uniform(
                -self.tumbling_range[1], self.tumbling_range[1], size=3
            )
            for i in range(self.NUM_DRONES):
                self.INIT_RPYS[i] = np.random.uniform(-0.1, 0.1, 3)

        return obs, info

    def _computeTruncated(self):
        state = self._getDroneStateVector(0)
        if (abs(state[0]) > 5.0 or abs(state[1]) > 5.0 or state[2] < 0.1 or state[2] > 10.0 or
            abs(state[7]) > 0.9 or abs(state[8]) > 0.9):
            return True
        if self.step_counter / self.PYB_FREQ > self.EPISODE_LEN_SEC:
            return True
        return False

# --- Find latest model ---
def find_latest_model():
    model_files = [
        "./models/phase_4_tumbling_curriculum.zip",
        "./models/phase_3_tumbling_curriculum.zip",
        "./models/phase_2_tumbling_curriculum.zip",
        "./models/phase_1_tumbling_curriculum.zip",
    ]
    for path in model_files:
        if os.path.exists(path):
            print(f"Found model: {path}")
            return path

    # Fallback: look for any recent model files
    if os.path.exists("./models/"):
        all_models = []
        for root, dirs, files in os.walk("./models/"):
            for file in files:
                if file.endswith(".zip"):
                    full_path = os.path.join(root, file)
                    all_models.append((os.path.getmtime(full_path), full_path))

        if all_models:
            latest_model = sorted(all_models, reverse=True)[0][1]
            print(f"Using most recent model: {latest_model}")
            return latest_model

    raise FileNotFoundError("No trained models found!")

model_path = find_latest_model()

print(f"Loading model from: {model_path}")
dummy_env = CurriculumHoverAviary(
    wind_speed_range=(0, 0),
    wind_angle_range=(0, 0),
    payload_range=(0, 0),
    tumbling_range=(0, 0),
    act=ActionType.PID,
    obs=ObservationType.KIN,
    gui=False,
    record=False
)
dummy_env = Monitor(dummy_env)
model = PPO.load(model_path, env=dummy_env)
dummy_env.close()

# --- Widgets ---
wind_speed_slider = widgets.FloatSlider(min=0, max=30, step=0.5, value=5, description='Wind Speed (m/s):')
wind_angle_slider = widgets.FloatSlider(min=0, max=359, step=1, value=90, description='Wind Angle (deg):')
payload_slider = widgets.FloatSlider(min=0, max=1000, step=10, value=500, description='Payload (g):')
ang_vel_x_text = widgets.FloatText(value=1.0, description='Ang Vel X (rad/s):')
ang_vel_y_text = widgets.FloatText(value=1.0, description='Ang Vel Y (rad/s):')
ang_vel_z_text = widgets.FloatText(value=0.5, description='Ang Vel Z (rad/s):')
run_button = widgets.Button(description="Run Simulation", button_style='success')
output_area = widgets.Output()

# --- Simulation function ---
def run_simulation(wind_speed, wind_angle, payload, ang_vel_x, ang_vel_y, ang_vel_z):
    with output_area:
        output_area.clear_output()
        print("Running simulation...")

        env = CurriculumHoverAviary(
            wind_speed_range=(wind_speed, wind_speed),
            wind_angle_range=(wind_angle, wind_angle),
            payload_range=(payload, payload),
            tumbling_range=(0, 0),
            gui=True,
            act=ActionType.PID,
            obs=ObservationType.KIN,
            record=False
        )
        env = Monitor(env)
        obs, info = env.reset()

        drone_id = env.unwrapped.DRONE_IDS[0]
        client = env.unwrapped.CLIENT
        p.resetBasePositionAndOrientation(drone_id, [0,0,9.5], [0,0,0,1], physicsClientId=client)
        p.resetBaseVelocity(drone_id, angularVelocity=[ang_vel_x, ang_vel_y, ang_vel_z], physicsClientId=client)
        p.stepSimulation(physicsClientId=client)
        env.unwrapped._updateAndStoreKinematicInformation()

        done = False
        total_reward = 0
        episode_length = 0
        frames = []
        time_steps, heights, angular_velocities = [], [], []

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            episode_length += 1

            state = env.unwrapped._getDroneStateVector(0)
            time_steps.append(episode_length * env.unwrapped.CTRL_TIMESTEP)
            heights.append(state[2])
            angular_velocities.append(state[13:16])

            try:
                drone_pos = state[0:3]
                img = p.getCameraImage(
                    width=1024, height=768,
                    viewMatrix=p.computeViewMatrixFromYawPitchRoll(
                        cameraTargetPosition=drone_pos,
                        distance=1.0, yaw=45, pitch=-15, roll=0, upAxisIndex=2),
                    projectionMatrix=p.computeProjectionMatrixFOV(
                        fov=60, aspect=1024/768, nearVal=0.1, farVal=100),
                    renderer=p.ER_BULLET_HARDWARE_OPENGL)
                rgba = np.array(img[2]).reshape(img[1], img[0], 4)
                frames.append(rgba[:, :, :3])
            except Exception as e:
                print(f"Frame capture warning: {e}")

            time.sleep(0.02)

        print(f"Simulation complete: Reward={total_reward:.2f}, Length={episode_length} steps")

        base_filename = f"sim_{int(wind_speed)}_{int(wind_angle)}_{int(payload)}_{ang_vel_x:.1f}_{ang_vel_y:.1f}_{ang_vel_z:.1f}"

        angular_velocities = np.array(angular_velocities)
        plt.figure(figsize=(12,6))
        plt.plot(time_steps, angular_velocities[:,0], 'r-', label='Roll Rate (X)')
        plt.plot(time_steps, angular_velocities[:,1], 'g-', label='Pitch Rate (Y)')
        plt.plot(time_steps, angular_velocities[:,2], 'b-', label='Yaw Rate (Z)')
        plt.axhline(0, color='k', linestyle='--', alpha=0.5)
        plt.xlabel('Time (s)')
        plt.ylabel('Angular Velocity (rad/s)')
        plt.title('Angular Stabilization')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        angular_plot = f"{base_filename}_angular.png"
        plt.savefig(angular_plot)
        plt.close()

        plt.figure(figsize=(12,6))
        plt.plot(time_steps, heights, 'purple', label='Height')
        plt.xlabel('Time (s)')
        plt.ylabel('Height (m)')
        plt.title('Height Stabilization')
        plt.ylim(0, 10.5)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        height_plot = f"{base_filename}_height.png"
        plt.savefig(height_plot)
        plt.close()

        if frames:
            gif_file = f"{base_filename}.gif"
            imageio.mimsave(gif_file, frames, duration=50)
            print(f"Saved files: {gif_file}, {angular_plot}, {height_plot}")
            ipy_display(IPyImage(filename=gif_file))
            ipy_display(IPyImage(filename=angular_plot))
            ipy_display(IPyImage(filename=height_plot))

        env.close()

# --- Button callback ---
def on_run_button_clicked(b):
    run_simulation(
        wind_speed_slider.value,
        wind_angle_slider.value,
        payload_slider.value,
        ang_vel_x_text.value,
        ang_vel_y_text.value,
        ang_vel_z_text.value
    )

# --- Widgets ---
wind_speed_slider = widgets.FloatSlider(min=0, max=30, step=0.5, value=5, description='Wind Speed (m/s):')
wind_angle_slider = widgets.FloatSlider(min=0, max=359, step=1, value=90, description='Wind Angle (deg):')
payload_slider = widgets.FloatSlider(min=0, max=1000, step=10, value=500, description='Payload (g):')
ang_vel_x_text = widgets.FloatText(value=1.0, description='Ang Vel X (rad/s):')
ang_vel_y_text = widgets.FloatText(value=1.0, description='Ang Vel Y (rad/s):')
ang_vel_z_text = widgets.FloatText(value=0.5, description='Ang Vel Z (rad/s):')
run_button = widgets.Button(description="Run Simulation", button_style='success')
output_area = widgets.Output()

run_button.on_click(on_run_button_clicked)

dashboard = VBox([
    widgets.HTML("<h3>Curriculum-Trained Drone Dashboard</h3>"),
    wind_speed_slider,
    wind_angle_slider,
    payload_slider,
    ang_vel_x_text,
    ang_vel_y_text,
    ang_vel_z_text,
    run_button,
    output_area
])

ipy_display(dashboard)

Using most recent model: ./models/Phase3_Tumbling_Fixed_final_20250711_153208.zip
Loading model from: ./models/Phase3_Tumbling_Fixed_final_20250711_153208.zip
[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 0.000000, km 0.000000,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


## INITIALIZE NOTEBOOK:

In [ ]:
#connect to drive:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Drone\ Stabilization\ GNN+Deep\ RL




In [ ]:
#import libraries:
%cd /content/drive/MyDrive/Drone\ Stabilization\ GNN+Deep\ RL
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
# MODELS:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader, Dataset
import torch.nn.functional as F


In [ ]:
# df=pd.read_csv('drone_scenarios.csv',sep=';')

In [ ]:
# df.head(2)

In [ ]:
#listing those which have greater than 1 less than 10:
# cat_cols=df.columns[(df.nunique() > 1) & (df.nunique() < 10)]
# cat_cols

In [ ]:
# single_state=df.columns[(df.nunique()==1)]
# single_state

In [ ]:
#dropping single state cols:
# df1=df.drop(columns=single_state,axis=1,inplace=True)

In [ ]:
# df.columns

In [ ]:
#plotting: plot for each flight (1 to 20) the change in the wind speed:
# columns_plot=df.columns[3:]

# flights_to_plot = [1, 2]  # or use df['flight'].unique()[:2]
# plt.figure(figsize=(20, 5))

# for col in columns_plot:
#     for idx, flight_id in enumerate(flights_to_plot, 1):
#       plt.subplot(1, 2, idx)
#       plt.plot(df[df['flight'] == flight_id][col])
#       plt.title(f'Flight {flight_id} - {col}')
#       plt.xlabel('Time Step')
#       plt.ylabel(f'{col}')

#     plt.tight_layout()
#     plt.show()

# Modeling GNN:
- normalizing data
- clearing data


In [ ]:
params=pd.read_csv('parameters.csv')

In [ ]:
params=params.drop(columns=['date',	'local_time','route'])


In [ ]:
params.head(2)

In [ ]:
#Sub-sampling:
# Define desired payload and speed values
payloads = [0, 250, 500]
speeds = [4, 6, 8, 10, 12]

# Sample per (payload, speed) combo
selected_rows = []
for p in payloads:
    for s in speeds:
        group = params[(params['payload'] == p) & (params['speed'] == s)]
        if len(group) >= 3:
            sample = group.sample(n=3, random_state=42)
        elif len(group) > 0:
            sample = group
        else:
            continue
        selected_rows.append(sample)

# Combine all selected flights
subset = pd.concat(selected_rows, ignore_index=True)
subset.to_csv("selected_flights.csv", index=False)

selected_ids = subset['flight'].tolist()
len(selected_ids)

In [ ]:
flight_dfs = []
for flight_id in selected_ids:
    df = pd.read_csv(f'flights/{flight_id}.csv')
    df['flight_id'] = flight_id

    # Add flight-level info
    meta = subset[subset['flight'] == flight_id].iloc[0]
    df['payload'] = meta['payload']
    df['speed'] = meta['speed']
    df['altitude'] = meta['altitude']  # if available in parameters.csv

    flight_dfs.append(df)

# Step 6: Combine all selected flights into one DataFrame
full_df = pd.concat(flight_dfs, ignore_index=True)

# Optional: move flight_id to first column
cols = ['flight_id'] + [col for col in full_df.columns if col != 'flight_id']
full_df = full_df[cols]


In [ ]:
full_df.shape

In [ ]:
full_df.columns

In [ ]:
full_df.to_csv("full_df.csv", index=False)

# 1) First LSTM for time-sequnece analysis:

## SAMPLED DATA FILE: df=pd.read_csv('full_df.csv')

In [ ]:
df=pd.read_csv('full_df.csv')

In [ ]:
from sklearn.preprocessing import StandardScaler
# Sort by flight_id and time
df = df.sort_values(by=['flight_id', 'time'])

# List of features to normalize and feed into LSTM
features = [
    'wind_speed', 'wind_angle',
    'battery_voltage', 'battery_current',
    'position_x', 'position_y', 'position_z',
    'orientation_x', 'orientation_y', 'orientation_z', 'orientation_w',
    'velocity_x', 'velocity_y', 'velocity_z',
    'angular_x', 'angular_y', 'angular_z',
    'linear_acceleration_x', 'linear_acceleration_y', 'linear_acceleration_z','payload', 'speed', 'altitude'
]
# Normalize features
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

In [ ]:
# Pick 3 example flight IDs
sample_ids = df['flight_id'].unique()[:3]

plt.figure(figsize=(15, 10))

for i, fid in enumerate(sample_ids):
    sub = df[df['flight_id'] == fid]

    plt.subplot(3, 1, i + 1)
    plt.plot(sub['time'], sub['velocity_z'], label='velocity_z')
    plt.plot(sub['time'], sub['angular_z'], label='angular_z')
    plt.plot(sub['time'], sub['altitude'], label='altitude')
    plt.title(f"Flight {fid}")
    plt.xlabel("Time (s)")
    plt.ylabel("Value (normalized)")
    plt.legend()

plt.tight_layout()
plt.show()


#Base LSTM : We’ll use a sliding window approach:

For each flight: break it into sequences of length seq_len (e.g., 20)

Each input X: shape (seq_len, n_features)

Each target Y: the next state (e.g., velocity_z at time t+1)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Select features (only dynamic ones for now)
sequence_features = [
    'wind_speed', 'wind_angle',
    'battery_voltage', 'battery_current',
    'position_x', 'position_y', 'position_z',
    'orientation_x', 'orientation_y', 'orientation_z', 'orientation_w',
    'velocity_x', 'velocity_y', 'velocity_z',
    'angular_x', 'angular_y', 'angular_z',
    'linear_acceleration_x', 'linear_acceleration_y',
    'linear_acceleration_z','payload', 'speed', 'altitude'
]

target_feature = 'velocity_z'  # Predict vertical velocity next step

SEQ_LEN = 20  # number of timesteps in the input sequence

# Custom PyTorch Dataset
class FlightDataset(Dataset):
    def __init__(self, df, seq_len=20):
        self.sequences = []
        grouped = df.groupby('flight_id')

        for _, group in grouped:
            group = group.sort_values('time')
            data = group[sequence_features].values
            target = group[target_feature].values

            for i in range(len(group) - seq_len):
                self.sequences.append((
                    data[i:i+seq_len],       # input sequence
                    target[i+seq_len]        # next timestep's target
                ))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        X, y = self.sequences[idx]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

dataset = FlightDataset(df, SEQ_LEN)
loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # lstm_out: [batch, seq, hidden]
        last_out = lstm_out[:, -1, :]  # take last time step
        out = self.fc(last_out)  # predict scalar
        return out.squeeze()


### Training:


In [ ]:
model = LSTMModel(input_size=len(sequence_features))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Training loop
for epoch in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}")


### SAVING MODEL:

In [ ]:
input_size = len(sequence_features)
hidden_size = 64   # or whatever you used
output_size = 1    # adjust if your target has more dimensions

In [ ]:
# Example path to save model
model_path = 'saved_models/lstm_stabilizer.pt'

# Create directory if needed

os.makedirs(os.path.dirname(model_path), exist_ok=True)

# Save model state_dict
torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': input_size,
    'hidden_size': hidden_size,
    'output_size': output_size
}, model_path)
print(f"Model saved to {model_path}")


## LOADING MODEL:

In [ ]:
#data file:
input_size = len(sequence_features)
hidden_size = 64   # or whatever you used
output_size = 1    # adjust if your target has more dimensions
df=pd.read_csv('full_df.csv')


# Load saved state
checkpoint = torch.load('saved_models/lstm_stabilizer.pt')

# Rebuild the same model
input_size = checkpoint['input_size']
hidden_size = checkpoint['hidden_size']
output_size = checkpoint['output_size']

model = LSTMModel(input_size, hidden_size, output_size)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Model loaded and ready for evaluation.")


In [ ]:
params=pd.read_csv('parameters.csv')

In [ ]:
# params

In [ ]:
df.columns

In [ ]:
df_test.columns

In [ ]:
test_df.columns

In [ ]:
# Load unseen flight
# Load test data for a specific flight
flight_no = 201
test_df = pd.read_csv(f"flights/{flight_no}.csv")
params = pd.read_csv("parameters.csv")
# Get corresponding parameters
param_row = params[params["flight"] == flight_no].iloc[0]

# Add constant columns to test_df
for col in ["speed", "payload", "altitude",'flight_id']:
  if col=='flight_id':
    test_df[col] = flight_no
  else:
    test_df[col] = param_row[col]

scaled_test = scaler.transform(test_df[features])
df_test = pd.DataFrame(scaled_test, columns=sequence_features)
df_test['flight_id'] = flight_no

In [ ]:

# Prepare sequence (same columns as training)
X_test = df_test[sequence_features].values[:SEQ_LEN]
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(0)  # shape: (1, seq_len, n_features)

# Predict
model.eval()
with torch.no_grad() :
    y_pred = model(X_test_tensor).item()

# Compare to true value
true_value = df_test[target_feature].iloc[SEQ_LEN]
print(f"Predicted: {y_pred:.4f}, Actual: {true_value:.4f}")
df_test['flight_id'] = flight_no

# Prepare sequence (same columns as training)


In [ ]:
X_seqs = []
y_targets = []

for i in range(len(df_test) - SEQ_LEN):
    seq_x = df_test[sequence_features].iloc[i:i+SEQ_LEN].values
    seq_y = df_test[target_feature].iloc[i+SEQ_LEN]  # next step

    X_seqs.append(seq_x)
    y_targets.append(seq_y)

X_test_tensor = torch.tensor(X_seqs, dtype=torch.float32)  # (num_sequences, SEQ_LEN, n_features)
y_true = torch.tensor(y_targets, dtype=torch.float32)


In [ ]:
model.eval()
with torch.no_grad():
    y_preds = model(X_test_tensor).squeeze().numpy()


In [ ]:
import torch.nn.functional as F

# Make sure y_preds and y_true_tensor are tensors
if isinstance(y_preds, torch.Tensor):
    y_pred_tensor = y_preds.view(-1)  # Ensure it's 1D
else:
    y_pred_tensor = torch.tensor(y_preds, dtype=torch.float32)

if isinstance(y_true, torch.Tensor):
    y_true_tensor = y_true.view(-1)
else:
    y_true_tensor = torch.tensor(y_true, dtype=torch.float32)

# Compute element-wise MSE (no reduction)
mse_each_step = F.mse_loss(y_pred_tensor, y_true_tensor, reduction='none').numpy()


In [ ]:

# Step 4: Plot MSE over time
plt.figure(figsize=(10, 5))
plt.plot(range(SEQ_LEN, SEQ_LEN + len(mse_each_step)), mse_each_step, label='MSE at each step')

plt.xlabel('Time Step (index)')
plt.ylabel('MSE')
plt.title('Mean Squared Error at Each Time Step')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#plot velocity_z component of the df_test on top of mse values for each time step:
plt.figure(figsize=(10, 6))
plt.plot(test_df['time'], df_test['velocity_z'], label='Actual', color='blue')
plt.plot(test_df['time'][SEQ_LEN:], mse_each_step, label='MSE', color='red')
plt.show()



## Generating Action vs State Features:


In [ ]:
df=pd.read_csv('full_df.csv')
df.columns

In [ ]:
from scipy.spatial.transform import Rotation as R

def compute_euler_angles(df):
    r = R.from_quat(df[['orientation_x', 'orientation_y', 'orientation_z', 'orientation_w']].values)
    euler = r.as_euler('xyz', degrees=False)  # radians
    df['roll'], df['pitch'], df['yaw'] = euler[:, 0], euler[:, 1], euler[:, 2]
    return df

In [ ]:
compute_euler_angles(df)

In [ ]:
#Also compute throttle using velocity and accelerationg:
g = 9.81
df['throttle_proxy'] = df['linear_acceleration_z'] + g

In [ ]:
df.head(2)
df.to_csv("full_df_actions.csv", index=False)

In [ ]:
!pip install gym



# Step1: Behavioral Cloning BC model: Supervised Learning:
 Learning from data:a kind of multi model regression
 We will then use the trained weights in the RL model

 - I will pull in sample data that has sample from different payload and speeds and exldue flight, time,and only from those r1-r7 and h mode..



In [ ]:
#import libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# MODELS:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader, Dataset
import torch.nn.functional as F


In [ ]:
os.getcwd()

In [ ]:
# #connect to drive:
# from google.colab import drive
# drive.mount('/content/drive')

%cd /content/drive/MyDrive/Drone\ Stabilization\ GNN+Deep\ RL



In [14]:
df=pd.read_csv('flights.csv',on_bad_lines='skip')

/tmp/ipython-input-14-1929727089.py:1: DtypeWarning: Columns (1,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('flights.csv',on_bad_lines='skip')


In [ ]:
import pandas as pd

# 1. Read the CSV (skip bad lines)
df = pd.read_csv('flights.csv', on_bad_lines='skip', low_memory=False)

# 2. Filter for R1–R7 and H routes
valid_routes = ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'H']
df_filtered = df[df['route'].isin(valid_routes)].copy()

# 3. Ensure payload and speed are numeric
df_filtered['payload'] = pd.to_numeric(df_filtered['payload'], errors='coerce')
df_filtered['speed'] = pd.to_numeric(df_filtered['speed'], errors='coerce')

# 4. Reset index and keep original index for tracking
df_filtered = df_filtered.reset_index(drop=False).rename(columns={'index': 'orig_idx'})

# 5. Group by (payload, speed)
pairs = df_filtered.groupby(['payload', 'speed'])

n_pairs = pairs.ngroups
samples_per_pair = 70_000 // n_pairs

sampled_list = []
for _, group in pairs:
    n = min(len(group), samples_per_pair)
    sampled = group.sample(n=n, random_state=42)
    sampled_list.append(sampled)

df_sampled = pd.concat(sampled_list)

# 6. If less than 70,000, sample the remainder from unused rows
used_indices = set(df_sampled['orig_idx'])
unused = df_filtered[~df_filtered['orig_idx'].isin(used_indices)]

if len(df_sampled) < 70_000 and not unused.empty:
    remainder = 70_000 - len(df_sampled)
    extra = unused.sample(n=min(remainder, len(unused)), random_state=42)
    df_sampled = pd.concat([df_sampled, extra])

# 7. Drop helper column if you want
df_sampled = df_sampled.drop(columns=['orig_idx'])

print(df_sampled.shape)
print(df_sampled[['payload', 'speed', 'route']].value_counts())

In [ ]:
df_sampled.columns

In [ ]:
df_sampled=df_sampled.drop(columns=['flight', 'time','position_x', 'position_y','altitude', 'date',
       'time_day', 'route'])

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R
df_sampled = df_sampled.apply(pd.to_numeric, errors='coerce')# Drone specs
m = 0.027  # kg
t2w = 2.25
g = 9.81  # m/s^2

# Euler angles from quaternion
quats = df_sampled[['orientation_x', 'orientation_y', 'orientation_z', 'orientation_w']].values
eulers = R.from_quat(quats).as_euler('xyz', degrees=False)
df_sampled['roll'] = eulers[:, 0]
df_sampled['pitch'] = eulers[:, 1]
df_sampled['yaw'] = eulers[:, 2]

# Throttle proxy using drone specs
df_sampled['throttle_proxy'] = (df_sampled['linear_acceleration_z'] + g) / (t2w * g)
df_sampled['throttle_proxy'] = np.clip(df_sampled['throttle_proxy'], 0, 1)  # Optional: clip to [0, 1]



In [ ]:
df_sampled.to_csv("full_df.csv", index=False)


### MODEL 1

In [ ]:
df_sampled=pd.read_csv('full_df.csv')
state_features = [
    'wind_speed','wind_angle',
    'position_z','velocity_x','velocity_y','velocity_z',
    'angular_x','angular_y','angular_z',
    'payload','speed'
]
action_targets = ['roll','pitch','yaw','throttle_proxy']

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# ------------------------------------------------------------------
# 1.  CONFIG
# ------------------------------------------------------------------
BATCH_SIZE     = 256
LR             = 1e-3
WEIGHT_DECAY   = 1e-4          # L2 regularisation strength
NUM_EPOCHS     = 200
THROTTLE_W     = 1.0           # <-- emphasise throttle_proxy
PITCH_W        = 1.0
YAW_W          = 1.0
ROLL_W        = 1.0
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

state_features = [
    'wind_speed','wind_angle',
    'position_z','velocity_x','velocity_y','velocity_z',
    'angular_x','angular_y','angular_z',
    'payload','speed'
]
action_targets = ['roll','pitch','yaw','throttle_proxy']
n_state   = len(state_features)
n_action  = len(action_targets)

# ------------------------------------------------------------------
# 2.  DATA  (assumes df_sampled already loaded)
# ------------------------------------------------------------------
X = torch.tensor(df_sampled[state_features].values,  dtype=torch.float32)
y = torch.tensor(df_sampled[action_targets].values, dtype=torch.float32)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True)


# Fit scaler on train, apply to both train and test
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

# (Optional) Scale y as well if action targets have very different ranges
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

import joblib
joblib.dump(scaler_X, "scaler_X.pkl")
joblib.dump(scaler_y, "scaler_y.pkl")


# Convert to torch tensors
X_train_torch = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_torch  = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_train_torch = torch.tensor(y_train_scaled, dtype=torch.float32)
y_test_torch  = torch.tensor(y_test_scaled,  dtype=torch.float32)


train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train_torch, y_train_torch),
    batch_size=BATCH_SIZE, shuffle=True)

test_loader  = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_test_torch,  y_test_torch),
    batch_size=BATCH_SIZE, shuffle=False)

# ------------------------------------------------------------------
# 3.  MODEL
# ------------------------------------------------------------------
class PolicyNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.net(x)

model = PolicyNet(input_dim=n_state, output_dim=n_action).to(DEVICE)

# ------------------------------------------------------------------
# 4.  LOSS  (weighted per-action MSE)
# ------------------------------------------------------------------
# create a weight vector: [1, 1, 1, THROTTLE_W]
loss_weights = torch.tensor(
    [ROLL_W, PITCH_W, YAW_W, THROTTLE_W], dtype=torch.float32, device=DEVICE
)

def weighted_mse(outputs, targets):
    # shape: (batch, action_dim)
    mse_per_dim = (outputs - targets).pow(2)
    weighted    = mse_per_dim * loss_weights
    return weighted.mean()

# ------------------------------------------------------------------
# 5.  OPTIMISER  (Adam + L2 weight decay)
# ------------------------------------------------------------------
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:


# ------------------------------------------------------------------
# 6.  TRAIN LOOP
# ------------------------------------------------------------------
best_loss = float('inf')
for epoch in range(1, NUM_EPOCHS + 1):
    # ---- train ----------------------------------------------------
    model.train()
    running_train_loss = 0.0
    for states, actions in train_loader:
        states, actions = states.to(DEVICE), actions.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(states)
        loss    = weighted_mse(outputs, actions)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * states.size(0)

    train_loss = running_train_loss / len(train_loader.dataset)

    # ---- evaluate -------------------------------------------------
    model.eval()
    running_test_loss = 0.0
    # collect preds/targets for per-action MSE
    preds_list, targs_list = [], []

    with torch.no_grad():
        for states, actions in test_loader:
            states, actions = states.to(DEVICE), actions.to(DEVICE)
            outputs = model(states)

            loss = weighted_mse(outputs, actions)
            running_test_loss += loss.item() * states.size(0)

            preds_list.append(outputs.cpu().numpy())
            targs_list.append(actions.cpu().numpy())

    test_loss = running_test_loss / len(test_loader.dataset)

    # # ---- per-action MSE ------------------------------------------
    preds = np.concatenate(preds_list, axis=0)
    targs = np.concatenate(targs_list, axis=0)
    per_action_mse = np.mean((preds - targs) ** 2, axis=0)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} "
              f"Train {train_loss:.4f}  Test {test_loss:.4f}")
        for i, act in enumerate(action_targets):
            print(f"   MSE {act:15s}: {per_action_mse[i]:.4f}")

    # ---- checkpoint ---------------------------------------------
    if test_loss < best_loss:
        epoch_best=epoch
        best_loss = test_loss
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss'          : train_loss,
            'test_loss'           : test_loss,
            # 'per_action_mse'      : per_action_mse,
        }, 'best_model.pth')
print(f"✅ New best model saved at epoch {epoch_best} (Test Loss: {best_loss:.4f})")

In [ ]:
import torch, numpy as np, torch.nn as nn, torch.optim as optim

# ------------------------------------------------------------
# CONFIG ── tweak as you like
# ------------------------------------------------------------
ADD_EPOCHS   = 300          # how many *extra* epochs to try
PATIENCE     = 50           # stop if no improvement for this many epochs
MIN_DELTA    = 1e-4         # improvement threshold
THROTTLE_W   = 1     # keep your throttle weight
YAW_W        = 1
PITCH_W      = 1
ROLL_W=1
WEIGHT_DECAY = 1e-4
LR           = 5e-5
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

# ────────────────────────────────────────────────────────────
# 1. Re-create model & optimizer exactly as before
# ────────────────────────────────────────────────────────────
class PolicyNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    def forward(self, x): return self.net(x)

n_state   = len(state_features)      # from your earlier code
n_action  = len(action_targets)
model     = PolicyNet(n_state, n_action).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# weighted loss
#ADDED WEIGHT ON YAW:
w = torch.tensor([1,1,YAW_W,THROTTLE_W], dtype=torch.float32, device=DEVICE)
def weighted_mse(o,t): return ((o-t).pow(2)*w).mean()

# ────────────────────────────────────────────────────────────
# 2. Load checkpoint
# ────────────────────────────────────────────────────────────
ckpt = torch.load("best_model.pth", map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt['epoch'] + 1
best_loss   = ckpt['test_loss']
print(f"✔️  Reloaded checkpoint at epoch {ckpt['epoch']}  (best test loss = {best_loss:.4f})")

# ────────────────────────────────────────────────────────────
# 3. Resume training with EARLY STOPPING
# ────────────────────────────────────────────────────────────
epochs_no_improve = 0
for epoch in range(start_epoch, start_epoch + ADD_EPOCHS):
    # ---- TRAIN ---------------------------------------------------
    model.train()
    train_sum = 0
    for s,a in train_loader:
        s,a = s.to(DEVICE), a.to(DEVICE)
        optimizer.zero_grad()
        loss = weighted_mse(model(s), a)
        loss.backward(); optimizer.step()
        train_sum += loss.item()*s.size(0)
    train_loss = train_sum / len(train_loader.dataset)

    # ---- EVAL ----------------------------------------------------
    model.eval()
    test_sum, preds, targs = 0, [], []
    with torch.no_grad():
        for s,a in test_loader:
            s,a = s.to(DEVICE), a.to(DEVICE)
            out = model(s)
            test_sum += weighted_mse(out,a).item()*s.size(0)
            preds.append(out.cpu().numpy()); targs.append(a.cpu().numpy())
    test_loss = test_sum / len(test_loader.dataset)

    # ---- per-action report every 10 epochs -----------------------
    if epoch % 20 == 0:
        pa_mse = np.mean((np.vstack(preds)-np.vstack(targs))**2,0)
        print(f"Epoch {epoch:3d}  Train {train_loss:.4f}  Test {test_loss:.4f}  "
              f"Δ={best_loss-test_loss:+.5f}")
        for i,a in enumerate(action_targets):
            print(f"   {a:15s} MSE: {pa_mse[i]:.4f}")

    # ---- EARLY-STOP LOGIC ---------------------------------------
    if test_loss < best_loss - MIN_DELTA:
        best_loss = test_loss
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'test_loss': best_loss
        }, 'best_model.pth')
        print(f"✅  New best model saved at epoch {epoch} (Test {best_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"⏹️  Early stopping at epoch {epoch}. "
                  f"No improvement for {PATIENCE} consecutive epochs.\n"f'Best MSE= {best_loss}')
            break

### Cheking Prediction and Test Resutls:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# -------------------------------------------------------------
# 1️⃣  Put model in eval mode and get predictions in one shot
# -------------------------------------------------------------
model.eval()
with torch.no_grad():
    # X_train_scaled is the numpy array AFTER StandardScaler
    preds_scaled = model(
        torch.tensor(X_train_scaled, dtype=torch.float32, device=DEVICE)
    ).cpu().numpy()

# -------------------------------------------------------------
# 2️⃣  Inverse-transform to physical units
# -------------------------------------------------------------
preds_orig   = scaler_y.inverse_transform(preds_scaled)          # shape (N,4)
actual_orig  = scaler_y.inverse_transform(y_train_scaled)        # or use y_train

action_names = ['roll', 'pitch', 'yaw', 'throttle_proxy']

# -------------------------------------------------------------
# 3️⃣  Plot
# -------------------------------------------------------------
plt.figure(figsize=(16, 8))
for i, name in enumerate(action_names):
    plt.subplot(2, 2, i + 1)
    plt.plot(actual_orig[:, i],  label='Actual',    alpha=0.7)
    plt.plot(preds_orig[:,  i],  label='Predicted', alpha=0.7)
    plt.title(f'{name.capitalize()} – Train Set')
    plt.xlabel('Sample')
    plt.ylabel(name)
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# -------------------------------------------------------------
# 1️⃣  Put model in eval mode and get predictions in one shot
# -------------------------------------------------------------
model.eval()
with torch.no_grad():
    # X_train_scaled is the numpy array AFTER StandardScaler
    preds_scaled = model(
        torch.tensor(X_test_scaled, dtype=torch.float32, device=DEVICE)
    ).cpu().numpy()

# -------------------------------------------------------------
# 2️⃣  Inverse-transform to physical units
# -------------------------------------------------------------
preds_orig   = scaler_y.inverse_transform(preds_scaled)          # shape (N,4)
actual_orig  = scaler_y.inverse_transform(y_test_scaled)        # or use y_train

action_names = ['roll', 'pitch', 'yaw', 'throttle_proxy']

# -------------------------------------------------------------
# 3️⃣  Plot
# -------------------------------------------------------------
plt.figure(figsize=(16, 8))
for i, name in enumerate(action_names):
    plt.subplot(2, 2, i + 1)
    plt.plot(actual_orig[:, i],  label='Actual',    alpha=0.7)
    plt.plot(preds_orig[:,  i],  label='Predicted', alpha=0.7)
    plt.title(f'{name.capitalize()} – Test Set')
    plt.xlabel('Sample')
    plt.ylabel(name)
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
df_sampled.head()

# STEP2: GYM_Pybullet-Drones:

In [1]:
%cd /content/drive/MyDrive/Drone Stabilization GNN+Deep RL

/content/drive/MyDrive/Drone Stabilization GNN+Deep RL


In [15]:
%ls models/default*


In [2]:
!apt-get install -y xvfb
!pip install pybullet
!pip install gym
!git clone https://github.com/utiasDSL/gym-pybullet-drones.git
%cd gym-pybullet-drones
!pip install -e .
%cd ..

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.15).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 MB 6.8 MB/s eta 0:00:00
fatal: destination path 'gym-pybullet-drones' already exists and is not an empty directory.
/content/drive/MyDrive/Drone Stabilization GNN+Deep RL/gym-pybullet-drones
Obtaining file:///content/drive/MyDrive/Drone%20Stabilization%20GNN%2BDeep%20RL/gym-pybullet-drones
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB

/content/drive/MyDrive/Drone Stabilization GNN+Deep RL


In [1]:
%cd  /content/drive/MyDrive/Drone Stabilization GNN+Deep RL

/content/drive/MyDrive/Drone Stabilization GNN+Deep RL


In [ ]:
#import libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# MODELS:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader, Dataset
import torch.nn.functional as F

# RL Training:


### initial setups:

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import joblib
import os
from stable_baselines3 import PPO
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import MlpExtractor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


### Adjusting the scaler with observation size:

In [ ]:
import numpy as np

try:
    from gym_pybullet_drones.envs.single_agent_rl.hover_aviary import HoverAviary
except ModuleNotFoundError:
    from gym_pybullet_drones.envs.HoverAviary import HoverAviary

env = HoverAviary(gui=False)

obs_list = []

for _ in range(1000):
    obs, _ = env.reset()
    obs = obs.squeeze()
    context = {
        'wind_speed': np.random.uniform(0.0, 17.1),
        'wind_angle': np.random.uniform(0.0, 359.0),
        'payload':    np.random.uniform(0.0, 500.0),
        'speed':      np.random.uniform(4.0, 12.0),
    }
    obs_vec = np.array([
        context['wind_speed'],      # 0
        context['wind_angle'],      # 1
        obs[2],                     # 2: position_z
        obs[10],                    # 3: velocity_x
        obs[11],                    # 4: velocity_y
        obs[12],                    # 5: velocity_z
        obs[13],                    # 6: angular_x
        obs[14],                    # 7: angular_y
        obs[15],                    # 8: angular_z
        context['payload'],         # 9
        context['speed'],           # 10
    ], dtype=np.float32)
    obs_list.append(obs_vec)

obs_array = np.stack(obs_list)
print("Shape for scaler:", obs_array.shape)  # Should be (1000, 11)

In [ ]:
# Load the scaler you used for BC training
scaler_X = joblib.load("scaler_X.pkl")  # or whatever filename you used

# Now use scaler_X to transform your RL observations
obs_array_scaled = scaler_X.transform(obs_array)

# Extreme Conditions Training:

In [ ]:
import gymnasium as gym
import numpy as np

class ContextualAviaryWrapper(gym.Wrapper):
    def __init__(self, env, scaler_X, fixed_context=None):
        super().__init__(env)
        self.scaler_X = scaler_X
        self.fixed_context = fixed_context
        obs_dim = 11  # Now using 11 features
        self.observation_space = gym.spaces.Box(
            low=-5.0, high=5.0, shape=(obs_dim,), dtype=np.float32
        )



        self.action_space = env.action_space

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        if self.fixed_context is not None:
            self.context = self.fixed_context
        else:
            # Domain randomization: sample from a wide range, including extremes
            self.context = {
                'wind_speed': np.random.uniform(0.0, 30.0),    # wider than expert data
                'wind_angle': np.random.uniform(0.0, 359.0),
                'payload':    np.random.uniform(0.0, 1000.0),  # heavier payloads
                'speed':      np.random.uniform(0.0, 20.0),    # wider speed range
            }
        obs_scaled = self._build_obs(obs)
        return obs_scaled, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_scaled = self._build_obs(obs)
        return obs_scaled, reward, terminated, truncated, info

    def _build_obs(self, hover_obs):
        if hover_obs.ndim == 2:
            hover_obs = hover_obs[0]
        obs_vec = np.array([
            self.context['wind_speed'],
            self.context['wind_angle'],
            hover_obs[2],    # position_z
            hover_obs[10],   # velocity_x
            hover_obs[11],   # velocity_y
            hover_obs[12],   # velocity_z
            hover_obs[13],   # angular_x
            hover_obs[14],   # angular_y
            hover_obs[15],   # angular_z
            self.context['payload'],
            self.context['speed'],
        ], dtype=np.float32)
        obs_scaled = self.scaler_X.transform(obs_vec.reshape(1, -1)).squeeze().astype(np.float32)
        return obs_scaled

In [ ]:
import os
import joblib
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import MlpExtractor

# Load your scaler (fit on 11 features)
scaler_X = joblib.load("scaler_X.pkl")

# Environment factory
def make_env(seed):
    def _init():
        try:
            from gym_pybullet_drones.envs.single_agent_rl.hover_aviary import HoverAviary
        except ModuleNotFoundError:
            from gym_pybullet_drones.envs.HoverAviary import HoverAviary
        env = HoverAviary(gui=False)
        return ContextualAviaryWrapper(env, scaler_X)
    return _init

n_envs = 4
vec_env = DummyVecEnv([make_env(i) for i in range(n_envs)])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class BCPolicy(ActorCriticPolicy):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    def _build_mlp_extractor(self) -> None:
        self.mlp_extractor = MlpExtractor(
            self.features_dim, net_arch=self.net_arch,
            activation_fn=self.activation_fn, device=self.device
        )

policy_kwargs = dict(
    net_arch=dict(pi=[128, 64], vf=[128, 64]),
    activation_fn=torch.nn.ReLU
)

# === NEW LOG AND CHECKPOINT DIRECTORIES ===
log_dir = "./ppo_logs_3"
checkpoint_dir = "./ppo_checkpoints_3"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

# Callbacks
checkpoint_callback = CheckpointCallback(
    save_freq=int(100000/n_envs),  # Save after every rollout (4 envs × 2048 steps)
    save_path=checkpoint_dir,
    name_prefix="ppo_drone"
)

# Optional: Evaluation callback for early stopping/best model
eval_env = DummyVecEnv([make_env(999)])
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(log_dir, 'best_model'),
    log_path=os.path.join(log_dir, 'results'),
    eval_freq=10000,
    n_eval_episodes=5,
    deterministic=True,
    render=False,
    verbose=1
)

# PPO agent
ppo = PPO(
    BCPolicy,
    vec_env,
    learning_rate=3e-5,
    batch_size=256,
    n_steps=2048,
    gamma=0.99,
    clip_range=0.1,
    target_kl=0.01,
    ent_coef=0.001,
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    device=DEVICE,
    policy_kwargs=policy_kwargs,
    tensorboard_log=log_dir  # <--- TensorBoard logs here!
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ppo_logs_2/

In [ ]:
# Train from scratch
ppo.learn(
    total_timesteps=200_000,
    callback=[checkpoint_callback, eval_callback]
)

ppo.save("ppo_finetuned_context_3")

In [ ]:

%cd /content/drive/MyDrive/Drone Stabilization GNN+Deep RL

In [ ]:
load_timestep = 1500000

n_envs = 4  # Leave 1 core for system
print(f"Using {n_envs} parallel environments.")

vec_env = DummyVecEnv([make_env(i) for i in range(n_envs)])

#Update the file destinations... June23 starting from 1.5M
ppo = PPO.load(f"./ppo_checkpoints_3/ppo_drone_{load_timestep}_steps.zip", env=vec_env, tensorboard_log="./ppo_logs/",device="cuda")
ppo.learn(
    total_timesteps=1000000,  #how many more steps
    callback=[checkpoint_callback, eval_callback],
    reset_num_timesteps=False
)
ppo.save("ppo_finetuned_context_3")

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir ppo_logs

# TUMBLING TRAINING:

### Initial Tumbling:

In [ ]:
# !apt-get update > /dev/null
#     !apt-get install -y xvfb > /dev/null
#     !pip install pybullet gym stable-baselines3==2.2.1 # Specify SB3 version if needed
#     !rm -rf gym-pybullet-drones # Clean up previous clone if exists
#     !git clone https://github.com/utiasDSL/gym-pybullet-drones.git
#     %cd gym-pybullet-drones
#     !pip install -e .
#     %cd ..
#     print("\n--- IMPORTANT ---")
#     print("If you just ran installations, please RESTART YOUR RUNTIME (Runtime -> Restart runtime) NOW.")
#     print("After restarting, re-run the %cd command, then run the main training script.")
#     ```
#     **Crucially, if you run the installation cell, you MUST restart the runtime, then re-run the `%cd` command, and *then* run the main training script below.**

# ---

# ### **Main Training Script (All Phases Combined)**

# This script will perform the following in sequence:

# 1.  **Initial 2M Steps:** Training with `[-pi, +pi]` angular range, original reward, original learning rate.
# 2.  **Phase 1 (2M to 2.2M steps):** Continue training with fixed context, `[-pi, +pi]` range, halved learning rate, original reward.
# 3.  **Phase 2 (2.2M to 2.4M steps):** Continue training with fixed context, `[-pi, +pi]` range, halved learning rate, and the roll/pitch penalty (coefficient 0.1).
# 4.  **Phase 3 (2.4M to 2.6M steps):** Continue training with fixed context, `[-pi, +pi]` range, halved learning rate, and **original reward** (reverting the penalty as per our last discussion).

# ```python
# os.makedirs(checkpoint_dir, exist_ok=True)
# os.makedirs(log_dir, exist_ok=True)

# # Define policy kwargs matching the original policy's MLP structure
# policy_kwargs = dict(
#     net_arch=dict(pi=[128, 64], vf=[128, 64]),
#     activation_fn=torch.nn.ReLU
# )

# # --- INITIAL TRAINING (0 to 2M steps) ---
# # Configuration: [-pi, +pi] angular range, original reward, original learning rate, random context

# print("\n--- INITIAL TRAINING (0 to 2M steps) ---")
# print("Configuration: Random Context, [-pi, +pi] Range, Original LR, Original Reward")

# # Set global reward mode for this phase
# CURRENT_REWARD_MODE = "original"
# ROLL_PITCH_PENALTY_COEFF = 0.0

# # Create environments for initial training (random context, pi range)
# vec_env_initial = DummyVecEnv([make_env_tumbling(i, tumble_range=pi, fixed_context=None) for i in range(n_envs)])
# eval_env_initial = DummyVecEnv([make_env_tumbling(999, tumble_range=pi, fixed_context=None)])

# # Initialize PPO model (or load if resuming from even earlier)
# # If you have a model from before 0 steps (e.g., a BC model), you can load it here.
# # Otherwise, it will start from scratch.
# try:
#     # Attempt to load a pre-existing model if it exists (e.g., from a previous run that crashed)
#     initial_model_path = os.path.join(checkpoint_dir, 'ppo_drone_tumble_finetune_2000000_steps.zip')
#     if os.path.exists(initial_model_path):
#         ppo_initial = PPO.load(
#             initial_model_path,
#             env=vec_env_initial,
#             device="cuda" if torch.cuda.is_available() else "cpu",
#             tensorboard_log=log_dir,
#             learning_rate=3e-5, # Original learning rate
#             policy_kwargs=policy_kwargs
#         )
#         print(f"Resuming initial training from {initial_model_path}")
#     else:
#         raise FileNotFoundError # Force new model if checkpoint not found
# except FileNotFoundError:
#     print("No 2M step checkpoint found. Starting initial training from scratch.")
#     ppo_initial = PPO(
#         "MlpPolicy",
#         vec_env_initial,
#         verbose=1,
#         tensorboard_log=log_dir,
#         learning_rate=3e-5, # Original learning rate
#         policy_kwargs=policy_kwargs
#     )

# # Callbacks for initial training
# checkpoint_callback_initial = CheckpointCallback(
#     save_freq=int(100000 / n_envs), # Save every 100K steps
#     save_path=checkpoint_dir,
#     name_prefix="ppo_drone_tumble_finetune"
# )

# eval_callback_initial = EvalCallback(
#     eval_env_initial,
#     best_model_save_path=os.path.join(log_dir, 'best_model_initial_training'),
#     log_path=os.path.join(log_dir, 'results_initial_training'),
#     eval_freq=10000,
#     n_eval_episodes=5,
#     deterministic=True,
#     render=False,
#     verbose=1
# )

# # Train for 2M steps (or continue if loaded)
# ppo_initial.learn(
#     total_timesteps=2_000_000,
#     callback=[checkpoint_callback_initial, eval_callback_initial],
#     reset_num_timesteps=False # Continue from current timestep if loaded
# )
# ppo_initial.save("ppo_finetuned_tumbling_2M_initial_training")
# print("Initial training (0-2M steps) complete. Model saved.")


# # --- PHASE 1 (2M to 2.2M steps): Fixed Context, [-pi, +pi] Range, Halved LR, Original Reward ---

# print("\n--- PHASE 1 (2M to 2.2M steps): Fixed Context, [-pi, +pi] Range, Halved LR, Original Reward ---")

# # Set global reward mode for this phase
# CURRENT_REWARD_MODE = "original"
# ROLL_PITCH_PENALTY_COEFF = 0.0

# # Create environments for Phase 1 (fixed context, pi range)
# vec_env_phase1 = DummyVecEnv([make_env_tumbling(i, tumble_range=pi, fixed_context=fixed_context_for_run) for i in range(n_envs)])
# eval_env_phase1 = DummyVecEnv([make_env_tumbling(999, tumble_range=pi, fixed_context=fixed_context_for_run)])

# # Load the model from the end of initial training (or its best version)
# resume_checkpoint_path_phase1 = os.path.join(checkpoint_dir, 'ppo_drone_tumble_finetune_2000000_steps.zip')
# # Or if you want to load the 'best_model' from initial training:
# # resume_checkpoint_path_phase1 = os.path.join(log_dir, 'best_model_initial_training', 'best_model.zip')

# print(f"Attempting to resume training from: {resume_checkpoint_path_phase1}")
# ppo_phase1 = PPO.load(
#     resume_checkpoint_path_phase1,
#     env=vec_env_phase1,
#     device="cuda" if torch.cuda.is_available() else "cpu",
#     tensorboard_log=log_dir,
#     learning_rate=3e-5 / 2, # Halved learning rate
#     policy_kwargs=policy_kwargs
# )
# print("✅ Model loaded successfully for Phase 1. Starting training.")

# checkpoint_callback_phase1 = CheckpointCallback(
#     save_freq=int(50000 / n_envs), # Save every 50K steps
#     save_path=checkpoint_dir,
#     name_prefix="ppo_drone_tumble_finetune_fixed_context_pi_lr_half"
# )

# eval_callback_phase1 = EvalCallback(
#     eval_env_phase1,
#     best_model_save_path=os.path.join(log_dir, 'best_model_fixed_context_pi_lr_half'),
#     log_path=os.path.join(log_dir, 'results_fixed_context_pi_lr_half'),
#     eval_freq=10000,
#     n_eval_episodes=5,
#     deterministic=True,
#     render=False,
#     verbose=1
# )

# ppo_phase1.learn(
#     total_timesteps=200_000, # Train for 200K additional steps
#     callback=[checkpoint_callback_phase1, eval_callback_phase1],
#     reset_num_timesteps=False
# )
# ppo_phase1.save("ppo_finetuned_tumbling_2M200K_fixed_context_pi_lr_half")
# print("Phase 1 training complete. Model saved.")


# # --- PHASE 2 (2.2M to 2.4M steps): High Pitch Roll Penalty (coeff 0.1) ---
# # Configuration: [-pi, +pi] angular range, fixed context, halved LR, NEW reward with penalty

# print("\n--- PHASE 2 (2.2M to 2.4M steps): High Pitch Roll Penalty (coeff 0.1) ---")

# # Set global reward mode and coefficient for this phase
# CURRENT_REWARD_MODE = "with_roll_pitch_penalty"
# ROLL_PITCH_PENALTY_COEFF = 0.1

# # Environments are the same as Phase 1 (fixed context, pi range)
# # No need to re-create vec_env_phase1 and eval_env_phase1, just reuse them.

# # Load the model saved at the end of Phase 1
# resume_checkpoint_path_phase2 = "ppo_finetuned_tumbling_2M200K_fixed_context_pi_lr_half.zip"

# print(f"Attempting to resume training from: {resume_checkpoint_path_phase2}")
# ppo_phase2 = PPO.load(
#     resume_checkpoint_path_phase2,
#     env=vec_env_phase1, # Reuse the same environment setup
#     device="cuda" if torch.cuda.is_available() else "cpu",
#     tensorboard_log=log_dir,
#     learning_rate=3e-5 / 2, # Keep the halved learning rate
#     policy_kwargs=policy_kwargs
# )
# print("✅ Model loaded successfully for Phase 2. Starting training with new reward penalty.")

# checkpoint_callback_phase2 = CheckpointCallback(
#     save_freq=int(50000 / n_envs),
#     save_path=checkpoint_dir,
#     name_prefix="ppo_drone_tumble_finetune_roll_pitch_penalty_0_1"
# )

# eval_callback_phase2 = EvalCallback(
#     eval_env_phase1, # Reuse the same environment setup
#     best_model_save_path=os.path.join(log_dir, 'best_model_roll_pitch_penalty_0_1'),
#     log_path=os.path.join(log_dir, 'results_roll_pitch_penalty_0_1'),
#     eval_freq=10000,
#     n_eval_episodes=5,
#     deterministic=True,
#     render=False,
#     verbose=1
# )

# ppo_phase2.learn(
#     total_timesteps=200_000, # Train for 200K additional steps
#     callback=[checkpoint_callback_phase2, eval_callback_phase2],
#     reset_num_timesteps=False
# )
# ppo_phase2.save("ppo_finetuned_tumbling_2M400K_roll_pitch_penalty_0_1")
# print("Phase 2 training complete. Model saved.")


# # --- PHASE 3 (2.4M to 2.6M steps): Continue with "Good Configuration" (Original Reward) ---
# # Configuration: [-pi, +pi] angular range, fixed context, halved LR, ORIGINAL reward

# print("\n--- PHASE 3 (2.4M to 2.6M steps): Continue with Good Configuration (Original Reward) ---")

# # Set global reward mode back to original
# CURRENT_REWARD_MODE = "original"
# ROLL_PITCH_PENALTY_COEFF = 0.0 # This value won't be used

# # Environments are the same as Phase 1 and 2 (fixed context, pi range)
# # No need to re-create vec_env_phase1 and eval_env_phase1, just reuse them.

# # Load the best model from Phase 1 (before the roll/pitch penalty was introduced)
# # This is crucial to revert to the better performing model.
# resume_checkpoint_path_phase3 = os.path.join(log_dir, 'best_model_fixed_context_pi_lr_half', 'best_model.zip')

# print(f"Attempting to resume training from: {resume_checkpoint_path_phase3}")
# ppo_phase3 = PPO.load(
#     resume_checkpoint_path_phase3,
#     env=vec_env_phase1, # Reuse the same environment setup
#     device="cuda" if torch.cuda.is_available() else "cpu",
#     tensorboard_log=log_dir,
#     learning_rate=3e-5 / 2, # Keep halved learning rate
#     policy_kwargs=policy_kwargs
# )
# print("✅ Model loaded successfully for Phase 3. Starting training.")

# checkpoint_callback_phase3 = CheckpointCallback(
#     save_freq=int(50000 / n_envs),
#     save_path=checkpoint_dir,
#     name_prefix="ppo_drone_tumble_finetune_good_config_phase3"
# )

# eval_callback_phase3 = EvalCallback(
#     eval_env_phase1, # Reuse the same environment setup
#     best_model_save_path=os.path.join(log_dir, 'best_model_good_config_phase3'),
#     log_path=os.path.join(log_dir, 'results_good_config_phase3'),
#     eval_freq=10000,
#     n_eval_episodes=5,
#     deterministic=True,
#     render=False,
#     verbose=1
# )

# ppo_phase3.learn(
#     total_timesteps=200_000, # Train for 200K additional steps
#     callback=[checkpoint_callback_phase3, eval_callback_phase3],
#     reset_num_timesteps=False
# )
# ppo_phase3.save("ppo_finetuned_tumbling_2M600K_good_config_phase3")
# print("Phase 3 training complete. Model saved.")

# print("\nAll training phases completed.")

### Best configuration tumbling:

In [ ]:
# Main Training Script - Cell 1 (to be run after initial setup and runtime restart)

import os
import joblib
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym
import numpy as np
import math

# --- IMPORTANT: Ensure your project directory is set correctly ---
# This should be the first thing you do in your Colab session after mounting Drive
# and before running any other code that depends on files in this directory.
# If you haven't already, run: %cd /content/drive/MyDrive/Drone Stabilization GNN+Deep RL

# --- TumblingRewardAviary and TumblingAviaryWrapper Class Definitions ---
# These are the classes that define your custom environment and reward.

try:
    from gym_pybullet_drones.envs.single_agent_rl.hover_aviary import HoverAviary
except ModuleNotFoundError:
    # This path assumes gym-pybullet-drones is installed in the environment
    # and its modules are accessible.
    from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ActionType

pi = math.pi

# Global variable to control reward function behavior
# For this phase, we are using the "original" reward (no roll/pitch penalty)
CURRENT_REWARD_MODE = "original"
ROLL_PITCH_PENALTY_COEFF = 0.0 # This value won't be used if mode is "original"

class TumblingRewardAviary(HoverAviary):
    def __init__(self, *args, **kwargs):
        if 'act' not in kwargs:
            kwargs['act'] = ActionType.PID
        super().__init__(*args, **kwargs)

    def _computeReward(self):
        state = self._getDroneStateVector(0)
        ang_vel = state[13:16]  # angular velocities x,y,z
        ang_vel_norm = np.linalg.norm(ang_vel)
        alive_bonus = 1.0
        stability_reward = np.exp(-0.5 * ang_vel_norm)
        reward = alive_bonus + stability_reward

        # --- Conditional NEW REWARD COMPONENT: Penalize high roll/pitch angles ---
        # This block is currently inactive because CURRENT_REWARD_MODE is "original"
        if CURRENT_REWARD_MODE == "with_roll_pitch_penalty":
            # state[7] is roll, state[8] is pitch (Euler angles)
            roll_pitch_penalty = -ROLL_PITCH_PENALTY_COEFF * (np.abs(state[7]) + np.abs(state[8]))
            reward += roll_pitch_penalty # Add the new penalty

        return reward

    def _computeTruncated(self):
        state = self._getDroneStateVector(0)
        # Termination conditions: out of bounds or too much tilt
        if (abs(state[0]) > 1.5 or abs(state[1]) > 1.5 or state[2] < 0.1 or state[2] > 2.0 or
            abs(state[7]) > 0.9 or abs(state[8]) > 0.9): # Roll/Pitch > ~51 degrees
            return True
        # Episode length termination
        if self.step_counter / self.PYB_FREQ > self.EPISODE_LEN_SEC:
            return True
        return False

class TumblingAviaryWrapper(gym.Wrapper):
    def __init__(self, env, scaler_X=None, fixed_context=None, tumble_range=pi): # Default to pi for this run
        super().__init__(env)
        self.scaler_X = scaler_X
        self.fixed_context = fixed_context # This will be fixed_context_for_run
        self.tumble_range = tumble_range
        obs_dim = 11 # Dimension of the observation space after wrapping
        self.observation_space = gym.spaces.Box(low=-5.0, high=5.0, shape=(obs_dim,), dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs) # Call the base environment's reset
        import pybullet as p
        drone_id = self.env.unwrapped.DRONE_IDS[0]
        client = self.env.unwrapped.CLIENT

        # Ensure simulation steps to apply any initial conditions from base env
        p.stepSimulation(physicsClientId=client)

        # Set initial angular velocity for tumbling
        new_ang_vel = np.random.uniform(-self.tumble_range, self.tumble_range, size=3)
        p.resetBaseVelocity(drone_id, angularVelocity=new_ang_vel.tolist(), physicsClientId=client)

        # Crucial: Step simulation and update kinematic info AFTER setting velocity
        p.stepSimulation(physicsClientId=client)
        self.env.unwrapped._updateAndStoreKinematicInformation()

        # --- MODIFIED CONTEXT HANDLING ---
        # Always randomize wind, but use fixed payload/speed if provided
        self.context = {
            'wind_speed': np.random.uniform(0.0, 30.0),
            'wind_angle': np.random.uniform(0.0, 359.0),
            'payload':    self.fixed_context.get('payload', np.random.uniform(0.0, 1000.0)), # Use fixed if exists, else random
            'speed':      self.fixed_context.get('speed', np.random.uniform(0.0, 20.0)),     # Use fixed if exists, else random
        }
        # --- END MODIFIED CONTEXT HANDLING ---

        # Build the observation vector for the agent
        current_env_obs = self.env._getDroneStateVector(0)
        obs_scaled = self._build_obs(current_env_obs)
        return obs_scaled, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_scaled = self._build_obs(obs)
        return obs_scaled, reward, terminated, truncated, info

    def _build_obs(self, hover_obs):
        if hover_obs.ndim == 2:
             hover_obs = hover_obs[0]

        # Construct the observation vector for the agent
        obs_vec = np.array([
            self.context.get('wind_speed', 0.0),
            self.context.get('wind_angle', 0.0),
            hover_obs[2],      # Z position
            hover_obs[10],     # Quaternion X
            hover_obs[11],     # Quaternion Y
            hover_obs[12],     # Quaternion Z
            hover_obs[13],     # Angular Velocity X
            hover_obs[14],     # Angular Velocity Y
            hover_obs[15],     # Angular Velocity Z
            self.context.get('payload', 0.0),
            self.context.get('speed', 0.0),
        ], dtype=np.float32)

        # Scale the observation if a scaler is provided
        if self.scaler_X is not None:
            obs_scaled = self.scaler_X.transform(obs_vec.reshape(1, -1)).squeeze().astype(np.float32)
        else:
            obs_scaled = obs_vec.astype(np.float32)
        return obs_scaled

# --- End of class definitions ---


# Load scaler (assuming scaler_X.pkl is in the current working directory)
print("Loading scaler_X.pkl...")
scaler_X = joblib.load("scaler_X.pkl")
print("Scaler loaded.")

# Define the fixed context for this run (Payload and Speed fixed)
fixed_context_for_run = {
    'payload': 1000.0,
    'speed': 15.0,
}

# === Environment Factories ===
# This function creates a new environment instance for training/evaluation
def make_env_tumbling_pi(seed):
    def _init():
        env = TumblingRewardAviary(gui=False) # gui=False for training
        # Pass the fixed_context and pi tumble_range
        env = TumblingAviaryWrapper(env, scaler_X, fixed_context=fixed_context_for_run, tumble_range=pi)
        env = Monitor(env) # Wrap with Monitor for logging
        return env
    return _init

n_envs = 4 # Number of parallel environments for training

# === LOG AND CHECKPOINT DIRECTORIES ===
checkpoint_dir = "./ppo_checkpoints_tumble_finetune"
log_dir = "./ppo_logs_tumble_finetune"
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# Define policy kwargs matching the original policy's MLP structure
policy_kwargs = dict(
    net_arch=dict(pi=[128, 64], vf=[128, 64]),
    activation_fn=torch.nn.ReLU
)

# --- PHASE 3: Continue with "Good Configuration" (2.2M to 2.4M steps) ---
# Load the best model from Phase 1 (before the roll/pitch penalty was introduced)

# IMPORTANT: Verify this path! It should point to the best model saved at the end of Phase 1.
# This was the model that showed improved performance with halved LR and fixed context.
# If you have a model that already has 2M tumbling steps, use that path instead!
# resume_checkpoint_path = "./ppo_logs_tumble_finetune/best_model_fixed_context_pi_lr_half/best_model.zip"
resume_checkpoint_path ="/content/drive/MyDrive/Drone Stabilization GNN+Deep RL/ppo_logs_tumble_finetune/best_model_good_config_phase3/best_model.zip"
print(f"\n--- PHASE 3: Continue with Good Configuration (Original Reward, Fixed Context, [-pi, +pi], Halved LR) ---")
print(f"Attempting to resume training from: {resume_checkpoint_path}")

# Set up environments for training and evaluation
vec_env = DummyVecEnv([make_env_tumbling_pi(i) for i in range(n_envs)])
eval_env = DummyVecEnv([make_env_tumbling_pi(999)]) # Separate environment for evaluation

# Load the PPO model
ppo_model = PPO.load(
    resume_checkpoint_path,
    env=vec_env,
    device="cuda" if torch.cuda.is_available() else "cpu",
    tensorboard_log=log_dir,
    learning_rate=3e-5 / 2,  # Keep the halved learning rate
    policy_kwargs=policy_kwargs
)

print("✅ Model loaded successfully for Phase 3. Starting training.")

# Define Checkpoint and Eval Callbacks
checkpoint_callback = CheckpointCallback(
    save_freq=int(100000 / n_envs),  # Save every 50K steps (total timesteps)
    save_path=checkpoint_dir,
    name_prefix="ppo_drone_tumble_finetune_good_config_phase3" # New prefix for this phase's checkpoints
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(log_dir, 'best_model_good_config_phase3'), # New path for best model
    log_path=os.path.join(log_dir, 'results_good_config_phase3'), # New log path for this phase
    eval_freq=10000, # Evaluate every 10K steps
    n_eval_episodes=5, # Number of episodes for evaluation
    deterministic=True, # Use deterministic actions for evaluation
    render=False, # No rendering during evaluation
    verbose=1
)

# Start training for 200,000 additional timesteps
ppo_model.learn(
    total_timesteps=1000_000,
    callback=[checkpoint_callback, eval_callback],
    reset_num_timesteps=False # Crucial: continue training from current timestep count
)

# Save the final model after this phase
ppo_model.save("ppo_finetuned_tumbling_good_config_phase3_200K")
print("Phase 3 training complete. Model saved as ppo_finetuned_tumbling_good_config_phase3_200K.")

### Visualization:

In [ ]:
%cd /content/drive/MyDrive/Drone Stabilization GNN+Deep RL

In [24]:
!pip install pyvirtualdisplay

In [ ]:
# Cell 2: Visualization (Pre-Tumbling Model: ppo_drone_1500000_steps.zip with ContextualAviaryWrapper - VARYING CONTEXT & CORRECT GIF DURATION)

import os
import joblib
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym
import numpy as np
import math
import time
import imageio # For creating GIF/MP4
from IPython.display import HTML, Image as IPyImage # Corrected import for direct use

# --- PyVirtualDisplay Setup ---
from pyvirtualdisplay import Display
import pybullet as p # Import pybullet here for screenshot functionality

print("Starting virtual display...")
DISPLAY_WIDTH = 1024
DISPLAY_HEIGHT = 768
display = Display(visible=0, size=(DISPLAY_WIDTH, DISPLAY_HEIGHT))
display.start()
print("Virtual display started.")
# --- End PyVirtualDisplay Setup ---

print("Starting visualization script...")

# Change to your project directory
try:
    %cd /content/drive/MyDrive/Drone Stabilization GNN+Deep RL
    print("Changed directory to /content/drive/MyDrive/Drone Stabilization GNN+Deep RL")
except Exception as e:
    print(f"Error changing directory: {e}")
    print("Please ensure the path is correct and Google Drive is mounted.")
    raise

# --- Environment Class Definitions ---
try:
    from gym_pybullet_drones.envs.single_agent_rl.hover_aviary import HoverAviary
except ModuleNotFoundError:
    from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ActionType

pi = math.pi

# --- TumblingRewardAviary (still needed if other parts of your project use it) ---
CURRENT_REWARD_MODE = "original"
ROLL_PITCH_PENALTY_COEFF = 0.0

class TumblingRewardAviary(HoverAviary):
    def __init__(self, *args, **kwargs):
        if 'act' not in kwargs:
            kwargs['act'] = ActionType.PID
        print("DEBUG: TumblingRewardAviary __init__ called.")
        super().__init__(*args, **kwargs)
        print("DEBUG: super().__init__ completed.")

    def _computeReward(self):
        state = self._getDroneStateVector(0)
        ang_vel = state[13:16]
        ang_vel_norm = np.linalg.norm(ang_vel)
        alive_bonus = 1.0
        stability_reward = np.exp(-0.5 * ang_vel_norm)
        reward = alive_bonus + stability_reward

        if CURRENT_REWARD_MODE == "with_roll_pitch_penalty":
            roll_pitch_penalty = -ROLL_PITCH_PENALTY_COEFF * (np.abs(state[7]) + np.abs(state[8]))
            reward += roll_pitch_penalty

        return reward

    def _computeTruncated(self):
        state = self._getDroneStateVector(0)
        if (abs(state[0]) > 1.5 or abs(state[1]) > 1.5 or state[2] < 0.1 or state[2] > 2.0 or
            abs(state[7]) > 0.9 or abs(state[8]) > 0.9):
            return True
        if self.step_counter / self.PYB_FREQ > self.EPISODE_LEN_SEC:
            return True
        return False

# --- TumblingAviaryWrapper (still needed if other parts of your project use it) ---
class TumblingAviaryWrapper(gym.Wrapper):
    def __init__(self, env, scaler_X=None, fixed_context=None, tumble_range=pi):
        super().__init__(env)
        self.scaler_X = scaler_X
        self.fixed_context = fixed_context
        self.tumble_range = tumble_range
        obs_dim = 11
        self.observation_space = gym.spaces.Box(low=-5.0, high=5.0, shape=(obs_dim,), dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        drone_id = self.env.unwrapped.DRONE_IDS[0]
        client = self.env.unwrapped.CLIENT

        p.stepSimulation(physicsClientId=client)

        if self.tumble_range > 0:
            new_ang_vel = np.random.uniform(-self.tumble_range, self.tumble_range, size=3)
            p.resetBaseVelocity(drone_id, angularVelocity=new_ang_vel.tolist(), physicsClientId=client)
            print(f"DEBUG: Initial angular velocity set to: {new_ang_vel}")
        else:
            print("DEBUG: Initial angular velocity not set (tumble_range is 0).")

        p.stepSimulation(physicsClientId=client)
        self.env.unwrapped._updateAndStoreKinematicInformation()

        if self.fixed_context is not None:
            self.context = self.fixed_context
        else:
            self.context = {
                'wind_speed': np.random.uniform(0.0, 30.0),
                'wind_angle': np.random.uniform(0.0, 359.0),
                'payload':    np.random.uniform(0.0, 1000.0),
                'speed':      np.random.uniform(0.0, 20.0),
            }

        print(f"DEBUG: Context: Payload={self.context.get('payload', 0):.1f}g, Speed={self.context.get('speed', 0):.1f}m/s")

        current_env_obs = self.env._getDroneStateVector(0)
        obs_scaled = self._build_obs(current_env_obs)
        return obs_scaled, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_scaled = self._build_obs(obs)
        return obs_scaled, reward, terminated, truncated, info

    def _build_obs(self, hover_obs):
        if hover_obs.ndim == 2:
             hover_obs = hover_obs[0]

        # --- DEEP DEBUGGING PRINT ---
        # print(f"DEBUG: TumblingAviaryWrapper _build_obs - self.context: {self.context}") # Keep this commented for now
        # --- END DEEP DEBUGGING PRINT ---

        obs_vec = np.array([
            self.context.get('wind_speed', 0.0), # Using .get with default for safety
            self.context.get('wind_angle', 0.0), # Using .get with default for safety
            hover_obs[2],
            hover_obs[10],
            hover_obs[11],
            hover_obs[12],
            hover_obs[13],
            hover_obs[14],
            hover_obs[15],
            self.context.get('payload', 0.0),
            self.context.get('speed', 0.0),
        ], dtype=np.float32)

        if self.scaler_X is not None:
            obs_scaled = self.scaler_X.transform(obs_vec.reshape(1, -1)).squeeze().astype(np.float32)
        else:
            obs_scaled = obs_vec.astype(np.float32)
        return obs_scaled

# --- ContextualAviaryWrapper (THE ONE USED FOR PRE-TUMBLING TRAINING) ---
class ContextualAviaryWrapper(gym.Wrapper):
    def __init__(self, env, scaler_X, fixed_context=None):
        super().__init__(env)
        self.scaler_X = scaler_X
        self.fixed_context = fixed_context
        obs_dim = 11
        self.observation_space = gym.spaces.Box(
            low=-5.0, high=5.0, shape=(obs_dim,), dtype=np.float32
        )
        self.action_space = env.action_space

        # Initialize context here to ensure it always exists
        # If fixed_context is provided, use it directly. It MUST contain all keys.
        if self.fixed_context is not None:
            self.context = self.fixed_context # Directly assign the complete fixed_context
        else:
            # Default context values, will be randomized in reset
            self.context = {
                'wind_speed': 0.0,
                'wind_angle': 0.0,
                'payload':    0.0,
                'speed':      0.0,
            }
        print(f"DEBUG: ContextualAviaryWrapper __init__ - self.context initialized to: {self.context}")


    def reset(self, **kwargs):
        _obs_base, info = self.env.reset(**kwargs)

        # Update context for the new episode
        if self.fixed_context is not None:
            # If fixed_context is provided, use it directly (it's already complete from __init__)
            self.context = self.fixed_context
        else:
            # Otherwise, randomize
            self.context = {
                'wind_speed': np.random.uniform(0.0, 30.0),
                'wind_angle': np.random.uniform(0.0, 359.0),
                'payload':    np.random.uniform(0.0, 1000.0),
                'speed':      np.random.uniform(0.0, 20.0),
            }
        print(f"DEBUG: ContextualAviaryWrapper reset - self.context updated to: {self.context}")

        self.env.unwrapped._updateAndStoreKinematicInformation()

        current_raw_state = self.env._getDroneStateVector(0)

        obs_scaled = self._build_obs(current_raw_state)

        return obs_scaled, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_scaled = self._build_obs(obs)
        return obs_scaled, reward, terminated, truncated, info

    def _build_obs(self, hover_obs):
        if hover_obs.ndim == 2:
            hover_obs = hover_obs[0]

        # --- DEEP DEBUGGING PRINT ---
        print(f"DEBUG: ContextualAviaryWrapper _build_obs - self.context: {self.context}")
        # --- END DEEP DEBUGGING PRINT ---

        obs_vec = np.array([
            self.context['wind_speed'], # These should now always exist
            self.context['wind_angle'], # These should now always exist
            hover_obs[2],    # position_z
            hover_obs[10],   # velocity_x
            hover_obs[11],   # velocity_y
            hover_obs[12],   # velocity_z
            hover_obs[13],   # angular_x
            hover_obs[14],   # angular_y
            hover_obs[15],   # angular_z
            self.context['payload'],
            self.context['speed'],
        ], dtype=np.float32)
        if self.scaler_X is not None:
            obs_scaled = self.scaler_X.transform(obs_vec.reshape(1, -1)).squeeze().astype(np.float32)
        else:
            obs_scaled = obs_vec.astype(np.float32)
        return obs_scaled


# --- Configuration for Rendering ---
print("Attempting to load scaler_X.pkl for pre-tumbling model...")
try:
    scaler_X = joblib.load("scaler_X.pkl")
    print("scaler_X.pkl loaded successfully.")
except Exception as e:
    print(f"Error loading scaler_X.pkl: {e}")
    print("Please ensure 'scaler_X.pkl' exists in the current directory.")
    raise

# Fixed context for rendering (NOW WITH DIFFERENT VALUES)
fixed_context_for_render = {
    'wind_speed': 45.0,  # Moderate wind
    'wind_angle': 90.0,  # Wind from the side
    'payload':    1000.0, # Moderate payload
    'speed':      10.0,  # Moderate forward speed
}

# *** MODEL PATH FOR PRE-TUMBLING MODEL (CORRECTED) ***
model_path = "./ppo_checkpoints_3/ppo_drone_1500000_steps.zip"

print(f"Attempting to load model from: {model_path}")
try:
    def make_render_env():
        print("Creating rendering environment for pre-tumbling model with GUI...")
        env = HoverAviary(gui=True, record=False)
        env = ContextualAviaryWrapper(env, scaler_X, fixed_context=fixed_context_for_render)
        env = Monitor(env)
        print("Rendering environment created.")
        return env

    print("DEBUG: Calling make_render_env()...")
    render_env = make_render_env()
    print("DEBUG: make_render_env() completed. Now attempting to load PPO model...")

    model = PPO.load(model_path, env=render_env)
    print("Wrapping the env in a DummyVecEnv.")
    print("✅ Model loaded successfully. Starting visualization...")

    num_render_episodes = 1

    for i in range(num_render_episodes):
        obs, info = render_env.reset()
        done = False
        total_reward = 0
        episode_length = 0

        print(f"\n--- Episode {i+1} ---")

        frames = []

        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = render_env.step(action)
            done = terminated or truncated
            total_reward += reward
            episode_length += 1

            try:
                client_id = render_env.unwrapped.CLIENT
                drone_pos = render_env.unwrapped._getDroneStateVector(0)[0:3]

                img = p.getCameraImage(
                    width=DISPLAY_WIDTH,
                    height=DISPLAY_HEIGHT,
                    viewMatrix=p.computeViewMatrixFromYawPitchRoll(
                        cameraTargetPosition=drone_pos,
                        distance=0.5,
                        yaw=45, pitch=-15, roll=0, upAxisIndex=2
                    ),
                    projectionMatrix=p.computeProjectionMatrixFOV(
                        fov=60, aspect=DISPLAY_WIDTH/DISPLAY_HEIGHT, nearVal=0.1, farVal=100
                    ),
                    renderer=p.ER_BULLET_HARDWARE_OPENGL
                )
                rgba_image = np.array(img[2]).reshape(img[1], img[0], 4)
                rgb_image = rgba_image[:, :, :3]
                frames.append(rgb_image)

            except Exception as capture_e:
                print(f"WARNING: Could not capture image for GIF: {capture_e}")

            time.sleep(0.02)

        print(f"Episode {i+1} finished. Total Reward: {total_reward:.2f}, Length: {episode_length} steps.")

        if frames:
            gif_path = f"drone_animation_pre_tumble_1.5M_episode_{i+1}.gif"
            print(f"Saving GIF to {gif_path} with {len(frames)} frames...")

            # Calculate duration per frame for a 6-second GIF
            # Ensure episode_length is not zero to avoid division by zero
            gif_duration_ms = (12 * 1000) / episode_length if episode_length > 0 else 100 # Default to 100ms if no frames

            imageio.mimsave(gif_path, frames, duration=gif_duration_ms) # Changed fps to duration
            print(f"GIF saved. Attempting to display...")

            try:
                IPyImage(filename=gif_path)
                print("GIF displayed using IPython.display.Image.")
            except Exception as display_e:
                print(f"WARNING: Could not display GIF using IPython.display.Image: {display_e}")
                print("Trying HTML embedding as fallback...")
                HTML(f'<img src="{gif_path}">')
                print("GIF displayed using HTML embedding.")
        else:
            print("No frames captured for GIF.")

        time.sleep(2)

    render_env.close()
    print("\nVisualization complete.")

except Exception as e:
    print(f"An error occurred during model loading or rendering: {e}")
    print("Please check the model_path and ensure all dependencies are correctly installed and the environment setup is complete.")
finally:
    if 'display' in locals():
        display.stop()
        print("Virtual display stopped.")

### TensorBoard

In [ ]:
# prompt: render results from tensorboard

%load_ext tensorboard
%tensorboard --logdir ppo_logs_tumble_finetune


### Dashboard Draft:

In [ ]:
# Visualization Script - Cell 2 (with User Input and Custom Filename)

import os
import joblib
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym
import numpy as np
import math
import time
import imageio # For creating GIF/MP4
from IPython.display import HTML, Image as IPyImage # Corrected import for direct use

# --- PyVirtualDisplay Setup ---
from pyvirtualdisplay import Display
import pybullet as p # Import pybullet here for screenshot functionality

print("Starting virtual display...")
DISPLAY_WIDTH = 1024
DISPLAY_HEIGHT = 768
display = Display(visible=0, size=(DISPLAY_WIDTH, DISPLAY_HEIGHT))
display.start()
print("Virtual display started.")
# --- End PyVirtualDisplay Setup ---

print("Starting visualization script...")

# Change to your project directory
try:
    %cd /content/drive/MyDrive/Drone Stabilization GNN+Deep RL
    print("Changed directory to /content/drive/MyDrive/Drone Stabilization GNN+Deep RL")
except Exception as e:
    print(f"Error changing directory: {e}")
    print("Please ensure the path is correct and Google Drive is mounted.")
    raise

# --- Environment Class Definitions (Copied from Training Script for Self-Containment) ---
try:
    from gym_pybullet_drones.envs.single_agent_rl.hover_aviary import HoverAviary
except ModuleNotFoundError:
    from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ActionType

pi = math.pi

# Global variable to control reward function behavior
CURRENT_REWARD_MODE = "original" # Ensure this matches your training
ROLL_PITCH_PENALTY_COEFF = 0.0

class TumblingRewardAviary(HoverAviary):
    def __init__(self, *args, **kwargs):
        if 'act' not in kwargs:
            kwargs['act'] = ActionType.PID
        super().__init__(*args, **kwargs)

    def _computeReward(self):
        state = self._getDroneStateVector(0)
        ang_vel = state[13:16]
        ang_vel_norm = np.linalg.norm(ang_vel)
        alive_bonus = 1.0
        stability_reward = np.exp(-0.5 * ang_vel_norm)
        reward = alive_bonus + stability_reward

        if CURRENT_REWARD_MODE == "with_roll_pitch_penalty":
            roll_pitch_penalty = -ROLL_PITCH_PENALTY_COEFF * (np.abs(state[7]) + np.abs(state[8]))
            reward += roll_pitch_penalty

        return reward

    def _computeTruncated(self):
        state = self._getDroneStateVector(0)
        if (abs(state[0]) > 1.5 or abs(state[1]) > 1.5 or state[2] < 0.1 or state[2] > 2.0 or
            abs(state[7]) > 0.9 or abs(state[8]) > 0.9):
            return True
        if self.step_counter / self.PYB_FREQ > self.EPISODE_LEN_SEC:
            return True
        return False

class TumblingAviaryWrapper(gym.Wrapper):
    def __init__(self, env, scaler_X=None, fixed_context=None, tumble_range=pi, initial_angular_velocity=None):
        super().__init__(env)
        self.scaler_X = scaler_X
        self.fixed_context = fixed_context
        self.tumble_range = tumble_range
        self.initial_angular_velocity = initial_angular_velocity # New parameter for visualization
        obs_dim = 11
        self.observation_space = gym.spaces.Box(low=-5.0, high=5.0, shape=(obs_dim,), dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        import pybullet as p
        drone_id = self.env.unwrapped.DRONE_IDS[0]
        client = self.env.unwrapped.CLIENT

        p.stepSimulation(physicsClientId=client)

        # --- CUSTOM INITIAL ANGULAR VELOCITY FOR VISUALIZATION ---
        if self.initial_angular_velocity is not None:
            new_ang_vel = np.array(self.initial_angular_velocity)
            print(f"DEBUG: Initial angular velocity set to USER INPUT: {new_ang_vel}")
        elif self.tumble_range > 0:
            new_ang_vel = np.random.uniform(-self.tumble_range, self.tumble_range, size=3)
            print(f"DEBUG: Initial angular velocity set to RANDOM: {new_ang_vel}")
        else:
            new_ang_vel = np.array([0.0, 0.0, 0.0])
            print("DEBUG: Initial angular velocity not set (tumble_range is 0 and no user input).")

        p.resetBaseVelocity(drone_id, angularVelocity=new_ang_vel.tolist(), physicsClientId=client)

        p.stepSimulation(physicsClientId=client)
        self.env.unwrapped._updateAndStoreKinematicInformation()

        # --- CUSTOM CONTEXT FOR VISUALIZATION ---
        # If fixed_context is provided, use it directly for all context variables
        if self.fixed_context is not None:
            self.context = self.fixed_context
            print(f"DEBUG: Context set to USER INPUT: Wind Speed={self.context.get('wind_speed', 0):.1f}, Wind Angle={self.context.get('wind_angle', 0):.1f}, Payload={self.context.get('payload', 0):.1f}g, Speed={self.context.get('speed', 0):.1f}m/s")
        else:
            # Fallback to random context if fixed_context is not provided (shouldn't happen in this script)
            self.context = {
                'wind_speed': np.random.uniform(0.0, 30.0),
                'wind_angle': np.random.uniform(0.0, 359.0),
                'payload':    np.random.uniform(0.0, 1000.0),
                'speed':      np.random.uniform(0.0, 20.0),
            }
            print(f"DEBUG: Context set to RANDOM: Wind Speed={self.context.get('wind_speed', 0):.1f}, Wind Angle={self.context.get('wind_angle', 0):.1f}, Payload={self.context.get('payload', 0):.1f}g, Speed={self.context.get('speed', 0):.1f}m/s")


        current_env_obs = self.env._getDroneStateVector(0)
        obs_scaled = self._build_obs(current_env_obs)
        return obs_scaled, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_scaled = self._build_obs(obs)
        return obs_scaled, reward, terminated, truncated, info

    def _build_obs(self, hover_obs):
        if hover_obs.ndim == 2:
             hover_obs = hover_obs[0]

        obs_vec = np.array([
            self.context.get('wind_speed', 0.0),
            self.context.get('wind_angle', 0.0),
            hover_obs[2],
            hover_obs[10],
            hover_obs[11],
            hover_obs[12],
            hover_obs[13],
            hover_obs[14],
            hover_obs[15],
            self.context.get('payload', 0.0),
            self.context.get('speed', 0.0),
        ], dtype=np.float32)

        if self.scaler_X is not None:
            obs_scaled = self.scaler_X.transform(obs_vec.reshape(1, -1)).squeeze().astype(np.float32)
        else:
            obs_scaled = obs_vec.astype(np.float32)
        return obs_scaled

# --- End of class definitions ---


# --- Configuration for Rendering ---
print("Attempting to load scaler_X.pkl...")
try:
    scaler_X = joblib.load("scaler_X.pkl")
    print("scaler_X.pkl loaded successfully.")
except Exception as e:
    print(f"Error loading scaler_X.pkl: {e}")
    print("Please ensure 'scaler_X.pkl' exists in the current directory.")
    raise

# *** USER INPUT FOR VISUALIZATION ***
print("\n--- Enter Visualization Parameters ---")
try:
    user_wind_speed = float(input("Enter Wind Speed (0.0 to 30.0 m/s): "))
    user_wind_angle = float(input("Enter Wind Angle (0.0 to 359.0 degrees): "))
    user_payload = float(input("Enter Payload (0.0 to 1000.0 grams): "))
    user_ang_vel_x = float(input("Enter Initial Angular Velocity X (e.g., 5.0 rad/s): "))
    user_ang_vel_y = float(input("Enter Initial Angular Velocity Y (e.g., -5.0 rad/s): "))
    user_ang_vel_z = float(input("Enter Initial Angular Velocity Z (e.g., 0.0 rad/s): "))
except ValueError:
    print("Invalid input. Please enter numeric values.")
    display.stop()
    raise

user_fixed_context = {
    'wind_speed': user_wind_speed,
    'wind_angle': user_wind_angle,
    'payload':    user_payload,
    'speed':      15.0, # Keeping speed fixed as per your training setup
}
user_initial_angular_velocity = [user_ang_vel_x, user_ang_vel_y, user_ang_vel_z]

# *** MODEL PATH FOR YOUR FINAL BEST MODEL ***
# This path is taken from your provided code-1's resume_checkpoint_path
model_path = "/content/drive/MyDrive/Drone Stabilization GNN+Deep RL/ppo_logs_tumble_finetune/best_model_good_config_phase3/best_model.zip"

print(f"Attempting to load model from: {model_path}")
try:
    def make_render_env():
        print("Creating rendering environment with GUI and user-defined context/angular velocity...")
        env = TumblingRewardAviary(gui=True, record=False) # gui=True for visualization
        # Pass user-defined context and initial angular velocity
        env = TumblingAviaryWrapper(env, scaler_X,
                                    fixed_context=user_fixed_context,
                                    tumble_range=0, # Set tumble_range to 0 so it doesn't override user_initial_angular_velocity
                                    initial_angular_velocity=user_initial_angular_velocity)
        env = Monitor(env)
        print("Rendering environment created.")
        return env

    print("DEBUG: Calling make_render_env()...")
    render_env = make_render_env()
    print("DEBUG: make_render_env() completed. Now attempting to load PPO model...")

    model = PPO.load(model_path, env=render_env)
    print("✅ Model loaded successfully. Starting visualization...")

    num_render_episodes = 1 # Always 1 for user input

    for i in range(num_render_episodes):
        obs, info = render_env.reset()
        done = False
        total_reward = 0
        episode_length = 0

        print(f"\n--- Episode {i+1} ---")

        frames = []

        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = render_env.step(action)
            done = terminated or truncated
            total_reward += reward
            episode_length += 1

            try:
                client_id = render_env.unwrapped.CLIENT
                drone_pos = render_env.unwrapped._getDroneStateVector(0)[0:3]

                img = p.getCameraImage(
                    width=DISPLAY_WIDTH,
                    height=DISPLAY_HEIGHT,
                    viewMatrix=p.computeViewMatrixFromYawPitchRoll(
                        cameraTargetPosition=drone_pos,
                        distance=0.5, # Adjusted camera distance for closer view
                        yaw=45, pitch=-15, roll=0, upAxisIndex=2
                    ),
                    projectionMatrix=p.computeProjectionMatrixFOV(
                        fov=60, aspect=DISPLAY_WIDTH/DISPLAY_HEIGHT, nearVal=0.1, farVal=100
                    ),
                    renderer=p.ER_BULLET_HARDWARE_OPENGL
                )
                rgba_image = np.array(img[2]).reshape(img[1], img[0], 4)
                rgb_image = rgba_image[:, :, :3]
                frames.append(rgb_image)

            except Exception as capture_e:
                print(f"WARNING: Could not capture image for GIF: {capture_e}")

            time.sleep(0.02)

        print(f"Episode {i+1} finished. Total Reward: {total_reward:.2f}, Length: {episode_length} steps.")

        if frames:
            # Construct filename based on user inputs
            gif_filename = (f"w{int(user_wind_speed)}_wa{int(user_wind_angle)}_"
                            f"pl{int(user_payload)}_sp{int(user_fixed_context['speed'])}_"
                            f"ax{user_ang_vel_x:.1f}_ay{user_ang_vel_y:.1f}_az{user_ang_vel_z:.1f}.gif")

            print(f"Saving GIF to {gif_filename} with {len(frames)} frames...")

            # Calculate duration per frame for a 6-second GIF (EPISODE_LEN_SEC is 6)
            gif_duration_ms = (render_env.unwrapped.EPISODE_LEN_SEC * 1000) / episode_length if episode_length > 0 else 100

            imageio.mimsave(gif_filename, frames, duration=gif_duration_ms)
            print(f"GIF saved. Attempting to display...")

            try:
                IPyImage(filename=gif_path) # This should be gif_filename
                print("GIF displayed using IPython.display.Image.")
            except Exception as display_e:
                print(f"WARNING: Could not display GIF using IPython.display.Image: {display_e}")
                print("Trying HTML embedding as fallback...")
                HTML(f'<img src="{gif_filename}">')
                print("GIF displayed using HTML embedding.")
        else:
            print("No frames captured for GIF.")

        time.sleep(2)

    render_env.close()
    print("\nVisualization complete.")

except Exception as e:
    print(f"An error occurred during model loading or rendering: {e}")
    print("Please check the model_path and ensure all dependencies are correctly installed and the environment setup is complete.")
finally:
    if 'display' in locals():
        display.stop()
        print("Virtual display stopped.")

### WIDGETS:

In [19]:
# Interactive Dashboard for Drone Simulation Visualization (Corrected for Tumbling Model)

import os
import joblib
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym
import numpy as np
import math
import time
import imageio
from IPython.display import HTML, Image as IPyImage, display as ipy_display
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, HBox, VBox, FloatSlider, FloatText, Button, Output

# --- PyVirtualDisplay Setup ---
from pyvirtualdisplay import Display
import pybullet as p

print("Starting virtual display...")
display_instance = None # Global variable to hold the display instance
DISPLAY_WIDTH = 1024
DISPLAY_HEIGHT = 768

def start_virtual_display():
    global display_instance
    if display_instance is None:
        display_instance = Display(visible=0, size=(DISPLAY_WIDTH, DISPLAY_HEIGHT))
        display_instance.start()
        print("Virtual display started.")
    else:
        try:
            display_instance.start() # Attempt to restart if it was stopped
            print("Virtual display re-started (was already running or stopped).")
        except Exception:
            print("Virtual display already running.")


def stop_virtual_display():
    global display_instance
    if display_instance is not None:
        try:
            display_instance.stop()
            print("Virtual display stopped.")
        except Exception as e:
            print(f"Error stopping virtual display: {e}")
        finally:
            display_instance = None

start_virtual_display()
# --- End PyVirtualDisplay Setup ---

print("Starting dashboard script...")

# Change to your project directory
try:
    project_dir = "/content/drive/MyDrive/Drone Stabilization GNN+Deep RL"
    os.chdir(project_dir)
    print(f"Changed directory to {project_dir}")
except Exception as e:
    print(f"Error changing directory: {e}")
    print("Please ensure the path is correct and Google Drive is mounted.")
    stop_virtual_display()
    raise

# --- Environment Class Definitions (Self-contained for the dashboard) ---
# These are the classes relevant to the model trained with TumblingAviaryWrapper
try:
    from gym_pybullet_drones.envs.single_agent_rl.hover_aviary import HoverAviary
except ModuleNotFoundError:
    from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ActionType

pi = math.pi

CURRENT_REWARD_MODE = "original"
ROLL_PITCH_PENALTY_COEFF = 0.0

class TumblingRewardAviary(HoverAviary):
    def __init__(self, *args, **kwargs):
        if 'act' not in kwargs:
            kwargs['act'] = ActionType.PID
        super().__init__(*args, **kwargs)

    def _computeReward(self):
        state = self._getDroneStateVector(0)
        ang_vel = state[13:16]
        ang_vel_norm = np.linalg.norm(ang_vel)
        alive_bonus = 1.0
        stability_reward = np.exp(-0.5 * ang_vel_norm)
        reward = alive_bonus + stability_reward

        if CURRENT_REWARD_MODE == "with_roll_pitch_penalty":
            roll_pitch_penalty = -ROLL_PITCH_PENALTY_COEFF * (np.abs(state[7]) + np.abs(state[8]))
            reward += roll_pitch_penalty

        return reward

    def _computeTruncated(self):
        state = self._getDroneStateVector(0)
        if (abs(state[0]) > 1.5 or abs(state[1]) > 1.5 or state[2] < 0.1 or state[2] > 2.0 or
            abs(state[7]) > 0.9 or abs(state[8]) > 0.9):
            return True
        if self.step_counter / self.PYB_FREQ > self.EPISODE_LEN_SEC:
            return True
        return False

class TumblingAviaryWrapper(gym.Wrapper):
    def __init__(self, env, scaler_X=None, fixed_context=None, tumble_range=pi, initial_angular_velocity=None):
        super().__init__(env)
        self.scaler_X = scaler_X
        self.fixed_context = fixed_context
        self.tumble_range = tumble_range
        self.initial_angular_velocity = initial_angular_velocity
        obs_dim = 11
        self.observation_space = gym.spaces.Box(low=-5.0, high=5.0, shape=(obs_dim,), dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        drone_id = self.env.unwrapped.DRONE_IDS[0]
        client = self.env.unwrapped.CLIENT

        p.stepSimulation(physicsClientId=client)

        if self.initial_angular_velocity is not None:
            new_ang_vel = np.array(self.initial_angular_velocity)
        elif self.tumble_range > 0:
            new_ang_vel = np.random.uniform(-self.tumble_range, self.tumble_range, size=3)
        else:
            new_ang_vel = np.array([0.0, 0.0, 0.0])

        p.resetBaseVelocity(drone_id, angularVelocity=new_ang_vel.tolist(), physicsClientId=client)

        p.stepSimulation(physicsClientId=client)
        self.env.unwrapped._updateAndStoreKinematicInformation()

        if self.fixed_context is not None:
            self.context = self.fixed_context
        else:
            self.context = {
                'wind_speed': np.random.uniform(0.0, 30.0),
                'wind_angle': np.random.uniform(0.0, 359.0),
                'payload':    np.random.uniform(0.0, 1000.0),
                'speed':      np.random.uniform(0.0, 20.0),
            }

        current_env_obs = self.env._getDroneStateVector(0)
        obs_scaled = self._build_obs(current_env_obs)
        return obs_scaled, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_scaled = self._build_obs(obs)
        return obs_scaled, reward, terminated, truncated, info

    def _build_obs(self, hover_obs):
        if hover_obs.ndim == 2:
             hover_obs = hover_obs[0]

        obs_vec = np.array([
            self.context.get('wind_speed', 0.0),
            self.context.get('wind_angle', 0.0),
            hover_obs[2],
            hover_obs[10],
            hover_obs[11],
            hover_obs[12],
            hover_obs[13],
            hover_obs[14],
            hover_obs[15],
            self.context.get('payload', 0.0),
            self.context.get('speed', 0.0),
        ], dtype=np.float32)

        if self.scaler_X is not None:
            obs_scaled = self.scaler_X.transform(obs_vec.reshape(1, -1)).squeeze().astype(np.float32)
        else:
            obs_scaled = obs_vec.astype(np.float32)
        return obs_scaled

# --- End of class definitions ---

# --- Load Scaler and Model (once) ---
print("Loading scaler_X.pkl...")
try:
    scaler_X = joblib.load("scaler_X.pkl")
    print("scaler_X.pkl loaded successfully.")
except Exception as e:
    print(f"Error loading scaler_X.pkl: {e}")
    print("Please ensure 'scaler_X.pkl' exists in the current directory.")
    stop_virtual_display()
    raise

# *** MODEL PATH FOR YOUR FINAL BEST MODEL (Tumbling Finetune Model) ***
model_path = "/content/drive/MyDrive/Drone Stabilization GNN+Deep RL/ppo_logs_tumble_finetune/best_model_good_config_phase3/best_model.zip"

print(f"Attempting to load model from: {model_path}")
try:
    # Create a dummy environment that matches the observation space the model was trained on
    # This means wrapping it with TumblingAviaryWrapper, even if it's just for loading
    dummy_base_env = TumblingRewardAviary(gui=False)
    dummy_wrapped_env_for_loading = TumblingAviaryWrapper(dummy_base_env, scaler_X,
                                                          fixed_context={'wind_speed': 0.0, 'wind_angle': 0.0, 'payload': 0.0, 'speed': 0.0},
                                                          tumble_range=0, initial_angular_velocity=[0.0, 0.0, 0.0])

    model = PPO.load(model_path, env=dummy_wrapped_env_for_loading)
    dummy_wrapped_env_for_loading.close() # Close the dummy env
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"An error occurred during model loading: {e}")
    print("Please check the model_path and ensure all dependencies are correctly installed.")
    stop_virtual_display()
    raise

# --- Widget Definitions ---
style = {'description_width': 'initial'}
layout = Layout(width='300px')

wind_speed_slider = FloatSlider(min=0.0, max=30.0, step=1.0, value=15.0,
                                description='Wind Speed (m/s):', style=style, layout=layout)
wind_angle_slider = FloatSlider(min=0.0, max=359.0, step=1.0, value=90.0,
                                description='Wind Angle (degrees):', style=style, layout=layout)
payload_slider = FloatSlider(min=0.0, max=1000.0, step=10.0, value=500.0,
                             description='Payload (grams):', style=style, layout=layout)

ang_vel_x_text = FloatText(value=0.0, description='Ang Vel X (rad/s):', style=style, layout=layout)
ang_vel_y_text = FloatText(value=0.0, description='Ang Vel Y (rad/s):', style=style, layout=layout)
ang_vel_z_text = FloatText(value=0.0, description='Ang Vel Z (rad/s):', style=style, layout=layout)

run_button = Button(description="Run Simulation", button_style='success', layout=Layout(width='200px', height='40px'))
output_area = Output()

# --- Simulation Function ---
def run_simulation(wind_speed, wind_angle, payload, ang_vel_x, ang_vel_y, ang_vel_z):
    with output_area:
        output_area.clear_output()
        print("Running simulation...")

        fixed_context_for_render = {
            'wind_speed': wind_speed,
            'wind_angle': wind_angle,
            'payload':    payload,
            'speed':      15.0, # Keeping speed fixed as per your training setup
        }
        initial_angular_velocity = [ang_vel_x, ang_vel_y, ang_vel_z]

        render_env = None
        try:
            render_env = TumblingRewardAviary(gui=True, record=False)
            render_env = TumblingAviaryWrapper(render_env, scaler_X,
                                                fixed_context=fixed_context_for_render,
                                                tumble_range=0, # Set to 0 to use initial_angular_velocity
                                                initial_angular_velocity=initial_angular_velocity)
            render_env = Monitor(render_env)

            obs, info = render_env.reset()
            done = False
            total_reward = 0
            episode_length = 0
            frames = []

            print(f"Simulating with: Wind Speed={wind_speed:.1f}, Wind Angle={wind_angle:.1f}, Payload={payload:.1f}g, "
                  f"Initial Ang Vel=[{ang_vel_x:.1f}, {ang_vel_y:.1f}, {ang_vel_z:.1f}]")

            while not done:
                action, _states = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = render_env.step(action)
                done = terminated or truncated
                total_reward += reward
                episode_length += 1

                try:
                    # Capture image for GIF
                    client_id = render_env.unwrapped.CLIENT
                    drone_pos = render_env.unwrapped._getDroneStateVector(0)[0:3]

                    img = p.getCameraImage(
                        width=DISPLAY_WIDTH,
                        height=DISPLAY_HEIGHT,
                        viewMatrix=p.computeViewMatrixFromYawPitchRoll(
                            cameraTargetPosition=drone_pos,
                            distance=0.5,
                            yaw=45, pitch=-15, roll=0, upAxisIndex=2
                        ),
                        projectionMatrix=p.computeProjectionMatrixFOV(
                            fov=60, aspect=DISPLAY_WIDTH/DISPLAY_HEIGHT, nearVal=0.1, farVal=100
                        ),
                        renderer=p.ER_BULLET_HARDWARE_OPENGL
                    )
                    rgba_image = np.array(img[2]).reshape(img[1], img[0], 4)
                    rgb_image = rgba_image[:, :, :3]
                    frames.append(rgb_image)

                except Exception as capture_e:
                    print(f"WARNING: Could not capture image for GIF: {capture_e}")

                time.sleep(0.02) # Small delay for visualization

            print(f"\nSimulation Finished.")
            print(f"Total Reward: {total_reward:.2f}")
            print(f"Episode Length: {episode_length} steps")

            if frames:
                gif_filename = (f"sim_w{int(wind_speed)}_wa{int(wind_angle)}_"
                                f"pl{int(payload)}_ax{ang_vel_x:.1f}_ay{ang_vel_y:.1f}_az{ang_vel_z:.1f}.gif")

                print(f"Saving GIF to {gif_filename} with {len(frames)} frames...")

                # Calculate duration per frame for a 6-second GIF (EPISODE_LEN_SEC is 6)
                gif_duration_ms = (render_env.unwrapped.EPISODE_LEN_SEC * 1000) / episode_length if episode_length > 0 else 100

                imageio.mimsave(gif_filename, frames, duration=gif_duration_ms)
                print(f"GIF saved. Displaying...")

                ipy_display(IPyImage(filename=gif_filename))
            else:
                print("No frames captured for GIF.")

        except Exception as e:
            print(f"An error occurred during simulation: {e}")
        finally:
            if render_env:
                render_env.close()
            print("Environment closed.")

# --- Connect Button to Function ---
def on_button_click(b):
    run_simulation(wind_speed_slider.value,
                   wind_angle_slider.value,
                   payload_slider.value,
                   ang_vel_x_text.value,
                   ang_vel_y_text.value,
                   ang_vel_z_text.value)

run_button.on_click(on_button_click)

# --- Dashboard Layout ---
input_widgets = VBox([
    wind_speed_slider,
    wind_angle_slider,
    payload_slider,
    ang_vel_x_text,
    ang_vel_y_text,
    ang_vel_z_text,
    run_button
])

dashboard = VBox([
    widgets.HTML("<h3>Drone Simulation Dashboard</h3>"),
    input_widgets,
    output_area
])

ipy_display(dashboard)

Starting virtual display...
Virtual display started.
Starting dashboard script...
Changed directory to /content/drive/MyDrive/Drone Stabilization GNN+Deep RL
Loading scaler_X.pkl...
scaler_X.pkl loaded successfully.
Attempting to load model from: /content/drive/MyDrive/Drone Stabilization GNN+Deep RL/ppo_logs_tumble_finetune/best_model_good_config_phase3/best_model.zip
[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 0.000000, km 0.000000,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
✅ Model loaded successfully.


### DEPLOY MODEL:

In [18]:
# Install necessary libraries for ONNX export and inference
# !pip install onnx onnxruntime

# Step-by-Step: Exporting your Stable Baselines3 PPO Model to ONNX (Final Refined Custom Wrapper with Action Scaling)

import os
import joblib
import torch
from stable_baselines3 import PPO
import gymnasium as gym
import numpy as np
import math
from torch import nn # Import nn for creating a wrapper module

# For ONNX export and inference
import onnxruntime as ort


# --- IMPORTANT: Ensure your project directory is set correctly ---
try:
    project_dir = "/content/drive/MyDrive/Drone Stabilization GNN+Deep RL"
    os.chdir(project_dir)
    print(f"Changed directory to {project_dir}")
except Exception as e:
    print(f"Error changing directory: {e}")
    print("Please ensure the path is correct and Google Drive is mounted.")
    raise

# --- Environment Class Definitions (Needed for dummy env to load model) ---
try:
    from gym_pybullet_drones.envs.single_agent_rl.hover_aviary import HoverAviary
except ModuleNotFoundError:
    from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ActionType

pi = math.pi

CURRENT_REWARD_MODE = "original"
ROLL_PITCH_PENALTY_COEFF = 0.0

class TumblingRewardAviary(HoverAviary):
    def __init__(self, *args, **kwargs):
        if 'act' not in kwargs:
            kwargs['act'] = ActionType.PID
        super().__init__(*args, **kwargs)

    def _computeReward(self):
        state = self._getDroneStateVector(0)
        ang_vel = state[13:16]
        ang_vel_norm = np.linalg.norm(ang_vel)
        alive_bonus = 1.0
        stability_reward = np.exp(-0.5 * ang_vel_norm)
        reward = alive_bonus + stability_reward

        if CURRENT_REWARD_MODE == "with_roll_pitch_penalty":
            roll_pitch_penalty = -ROLL_PITCH_PENALTY_COEFF * (np.abs(state[7]) + np.abs(state[8]))
            reward += roll_pitch_penalty

        return reward

    def _computeTruncated(self):
        state = self._getDroneStateVector(0)
        if (abs(state[0]) > 1.5 or abs(state[1]) > 1.5 or state[2] < 0.1 or state[2] > 2.0 or
            abs(state[7]) > 0.9 or abs(state[8]) > 0.9):
            return True
        if self.step_counter / self.PYB_FREQ > self.EPISODE_LEN_SEC:
            return True
        return False

class TumblingAviaryWrapper(gym.Wrapper):
    def __init__(self, env, scaler_X=None, fixed_context=None, tumble_range=pi, initial_angular_velocity=None):
        super().__init__(env)
        self.scaler_X = scaler_X
        self.fixed_context = fixed_context
        self.tumble_range = tumble_range
        self.initial_angular_velocity = initial_angular_velocity
        obs_dim = 11
        self.observation_space = gym.spaces.Box(low=-5.0, high=5.0, shape=(obs_dim,), dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        import pybullet as p
        drone_id = self.env.unwrapped.DRONE_IDS[0]
        client = self.env.unwrapped.CLIENT

        p.stepSimulation(physicsClientId=client)

        if self.initial_angular_velocity is not None:
            new_ang_vel = np.array(self.initial_angular_velocity)
        elif self.tumble_range > 0:
            new_ang_vel = np.random.uniform(-self.tumble_range, self.tumble_range, size=3)
        else:
            new_ang_vel = np.array([0.0, 0.0, 0.0])

        p.resetBaseVelocity(drone_id, angularVelocity=new_ang_vel.tolist(), physicsClientId=client)

        p.stepSimulation(physicsClientId=client)
        self.env.unwrapped._updateAndStoreKinematicInformation()

        if self.fixed_context is not None:
            self.context = self.fixed_context
        else:
            self.context = {
                'wind_speed': np.random.uniform(0.0, 30.0),
                'wind_angle': np.random.uniform(0.0, 359.0),
                'payload':    np.random.uniform(0.0, 1000.0),
                'speed':      np.random.uniform(0.0, 20.0),
            }

        current_env_obs = self.env._getDroneStateVector(0)
        obs_scaled = self._build_obs(current_env_obs)
        return obs_scaled, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_scaled = self._build_obs(obs)
        return obs_scaled, reward, terminated, truncated, info

    def _build_obs(self, hover_obs):
        if hover_obs.ndim == 2:
             hover_obs = hover_obs[0]

        obs_vec = np.array([
            self.context.get('wind_speed', 0.0),
            self.context.get('wind_angle', 0.0),
            hover_obs[2],
            hover_obs[10],
            hover_obs[11],
            hover_obs[12],
            hover_obs[13],
            hover_obs[14],
            hover_obs[15],
            self.context.get('payload', 0.0),
            self.context.get('speed', 0.0),
        ], dtype=np.float32)

        if self.scaler_X is not None:
            obs_scaled = self.scaler_X.transform(obs_vec.reshape(1, -1)).squeeze().astype(np.float32)
        else:
            obs_scaled = obs_vec.astype(np.float32)
        return obs_scaled

# --- End of class definitions ---

# --- Load Scaler (needed for observation preprocessing) ---
print("Loading scaler_X.pkl...")
try:
    scaler_X = joblib.load("scaler_X.pkl")
    print("scaler_X.pkl loaded successfully.")
except Exception as e:
    print(f"Error loading scaler_X.pkl: {e}")
    print("Please ensure 'scaler_X.pkl' exists in the current directory.")
    raise

# --- Step 1: Load Your Trained Model ---
model_path = "/content/drive/MyDrive/Drone Stabilization GNN+Deep RL/ppo_logs_tumble_finetune/best_model_good_config_phase3/best_model.zip"

print(f"\n--- Step 1: Loading PPO model from: {model_path} ---")
try:
    dummy_base_env = TumblingRewardAviary(gui=False)
    dummy_wrapped_env_for_loading = TumblingAviaryWrapper(dummy_base_env, scaler_X,
                                                          fixed_context={'wind_speed': 0.0, 'wind_angle': 0.0, 'payload': 0.0, 'speed': 0.0},
                                                          tumble_range=0, initial_angular_velocity=[0.0, 0.0, 0.0])

    ppo_model = PPO.load(model_path, env=dummy_wrapped_env_for_loading)
    dummy_wrapped_env_for_loading.close()
    print("✅ PPO model loaded successfully.")
except Exception as e:
    print(f"An error occurred during PPO model loading: {e}")
    print("Please check the model_path and ensure all dependencies are correctly installed.")
    raise

# --- Step 2: Export to ONNX using a refined custom wrapper with action scaling ---
onnx_model_path = "stabilization_policy_deterministic_final_onnx.onnx" # New filename

# Create a dummy input for ONNX export
dummy_input = torch.randn(1, ppo_model.observation_space.shape[0], dtype=torch.float32)

print(f"\n--- Step 2: Exporting model to ONNX: {onnx_model_path} ---")
try:
    # Get action space bounds for scaling
    action_low = torch.from_numpy(ppo_model.action_space.low).float()
    action_high = torch.from_numpy(ppo_model.action_space.high).float()

    # Define a custom ONNX-compatible wrapper that mimics the deterministic predict logic
    class OnnxableDeterministicPolicy(nn.Module):
        def __init__(self, policy_module, action_low, action_high):
            super().__init__()
            self.features_extractor = policy_module.features_extractor
            self.mlp_extractor = policy_module.mlp_extractor
            self.action_net = policy_module.action_net
            self.action_low = action_low
            self.action_high = action_high
            self.eval() # Ensure the wrapper is in evaluation mode

        def forward(self, obs: torch.Tensor) -> torch.Tensor:
            # 1. Extract features using the policy's feature extractor
            features = self.features_extractor(obs)

            # 2. Pass features through the MLP extractor (shared network)
            latent_pi, _ = self.mlp_extractor(features) # mlp_extractor returns (latent_pi, latent_vf)

            # 3. Pass latent_pi through the action network to get the raw action output
            raw_action_output = self.action_net(latent_pi)

            # 4. Apply Tanh activation (standard for continuous actions in SB3)
            actions = torch.tanh(raw_action_output)

            # 5. Scale actions to the environment's action space
            # This is the formula: action = low + (high - low) * ( (action_tanh + 1) / 2 )
            # Which simplifies to: action = action_tanh * (high - low) / 2 + (high + low) / 2
            scaled_actions = self.action_low + (self.action_high - self.action_low) * ((actions + 1) / 2)

            return scaled_actions

    # Instantiate the refined ONNX-compatible policy wrapper
    onnx_policy_wrapper = OnnxableDeterministicPolicy(ppo_model.policy.to("cpu"), action_low, action_high)
    onnx_policy_wrapper.eval() # Explicitly set to eval mode

    torch.onnx.export(
        onnx_policy_wrapper,
        dummy_input,
        onnx_model_path,
        input_names=['obs'],
        output_names=['action'],
        dynamic_axes={'obs': {0: 'batch_size'}, 'action': {0: 'batch_size'}}, # Allow dynamic batch size
        opset_version=11, # Common opset version
        do_constant_folding=True,
    )
    print("✅ Model successfully exported to ONNX.")
except Exception as e:
    print(f"Error exporting model to ONNX: {e}")
    print("This might be due to unsupported operations or complex policy structure.")
    raise

# --- Step 3: Load and Test the ONNX Model ---
print(f"\n--- Step 3: Loading and testing the ONNX model: {onnx_model_path} ---")
try:
    # Load the ONNX model
    onnx_model = onnx.load(onnx_model_path)
    onnx.checker.check_model(onnx_model) # Check if the model is valid
    print("✅ ONNX model loaded and checked successfully.")

    # Create an ONNX Runtime session
    ort_session = ort.InferenceSession(onnx_model_path)
    print("✅ ONNX Runtime session created.")

    # Get input and output names
    input_name = ort_session.get_inputs()[0].name
    output_name = ort_session.get_outputs()[0].name
    print(f"ONNX Input Name: {input_name}, Output Name: {output_name}")

    # Test with a sample observation (e.g., from your environment)
    dummy_raw_obs = np.array([
        15.0,      # wind_speed
        90.0,      # wind_angle
        1.0,       # Z position (hover target)
        0.0, 0.0, 0.0, # Quaternion (identity)
        0.1, 0.2, 0.3, # Angular Velocity (small, non-zero)
        500.0,     # Payload
        15.0       # Speed
    ], dtype=np.float32)

    # Scale the dummy observation using the loaded scaler
    dummy_scaled_obs = scaler_X.transform(dummy_raw_obs.reshape(1, -1)).squeeze().astype(np.float32)

    # ONNX Runtime expects inputs as a dictionary {input_name: numpy_array}
    # and the numpy array needs to have the batch dimension.
    ort_input = {input_name: dummy_scaled_obs.reshape(1, -1)}

    # Get action from original PPO model (deterministic)
    with torch.no_grad():
        original_action, _ = ppo_model.predict(dummy_scaled_obs, deterministic=True)

    # Get action from ONNX model
    ort_output = ort_session.run([output_name], ort_input)
    onnx_action = ort_output[0].squeeze() # Extract action and remove batch dimension

    print(f"Dummy Scaled Observation: {dummy_scaled_obs}")
    print(f"Action from original PPO model (deterministic): {original_action}")
    print(f"Action from ONNX model: {onnx_action}")

    # Check if actions are close
    if np.allclose(original_action, onnx_action, atol=1e-5):
        print("✅ Actions from original PPO model and ONNX model are very close!")
        print("Your deterministic model is successfully exported to ONNX and ready for standalone loading.")
    else:
        print("❌ Warning: Actions from original PPO model and ONNX model differ significantly.")
        print("This might indicate an issue with the ONNX export or model structure.")
        print(f"Absolute difference: {np.abs(original_action - onnx_action).max()}")
        print(f"Relative difference: {np.max(np.abs(original_action - onnx_action) / (np.abs(original_action) + 1e-8))}")

except Exception as e:
    print(f"An error occurred during ONNX model loading or testing: {e}")
    print("Please ensure 'onnxruntime' is installed and the ONNX model is valid.")
    raise

print("\nDeployment preparation complete. The 'stabilization_policy_deterministic_final_onnx.onnx' file is your standalone model.")
print("You can now load this '.onnx' file using ONNX Runtime in various environments (Python, C++, etc.).")

Changed directory to /content/drive/MyDrive/Drone Stabilization GNN+Deep RL
Loading scaler_X.pkl...
scaler_X.pkl loaded successfully.

--- Step 1: Loading PPO model from: /content/drive/MyDrive/Drone Stabilization GNN+Deep RL/ppo_logs_tumble_finetune/best_model_good_config_phase3/best_model.zip ---
[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 0.000000, km 0.000000,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
✅ PPO model loaded successfully.

--- Step 2: Exporting model to ONNX: stabilization_policy_deterministic_final_onnx.onnx ---
Error exporting model to ONNX: Module onnx is not installed!
This might be due to unsu

OnnxExporterError: Module onnx is not installed!

### MODEL on ONNX:

In [ ]:
%cd /content/drive/MyDrive/Drone Stabilization GNN+Deep RL

In [4]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.1 MB/s eta 0:00:00


In [16]:
import onnxruntime as ort
import numpy as np
import joblib # If you need to load the scaler

# --- Configuration ---
ONNX_MODEL_PATH = "stabilization_policy_deterministic_final_onnx.onnx"
SCALER_PATH = "scaler_X.pkl" # Path to your scaler

# --- Load Scaler (if needed for preprocessing observations) ---
try:
    scaler_X = joblib.load(SCALER_PATH)
    print("Scaler loaded successfully.")
except Exception as e:
    print(f"Error loading scaler: {e}")
    print("Ensure 'scaler_X.pkl' is in the correct directory or adjust SCALER_PATH.")
    exit()

# --- Load ONNX Runtime Session ---
try:
    ort_session = ort.InferenceSession(ONNX_MODEL_PATH)
    input_name = ort_session.get_inputs()[0].name
    output_name = ort_session.get_outputs()[0].name
    print(f"ONNX model loaded. Input: {input_name}, Output: {output_name}")
except Exception as e:
    print(f"Error loading ONNX model: {e}")
    print("Ensure 'stabilization_policy_deterministic_final_onnx.onnx' is in the correct directory.")
    exit()

# --- Example Inference ---
def get_action_from_onnx_model(raw_observation: np.ndarray) -> np.ndarray:
    """
    Preprocesses a raw observation, feeds it to the ONNX model, and returns the action.
    """
    # 1. Scale the raw observation
    # Ensure the input to scaler is 2D (batch_size, features)
    scaled_obs = scaler_X.transform(raw_observation.reshape(1, -1)).squeeze().astype(np.float32)

    # 2. Prepare input for ONNX Runtime (add batch dimension if not already present)
    ort_input = {input_name: scaled_obs.reshape(1, -1)}

    # 3. Run inference
    ort_output = ort_session.run([output_name], ort_input)

    # 4. Extract action and remove batch dimension
    action = ort_output[0].squeeze()

    return action

# --- Test with a dummy raw observation (example) ---
# This should be a raw observation from your environment, before scaling
dummy_raw_obs_for_inference = np.array([
    15.0,      # wind_speed
    90.0,      # wind_angle
    1.0,       # Z position (hover target)
    0.0, 0.0, 0.0, # Quaternion (identity)
    0.1, 0.2, 0.3, # Angular Velocity (small, non-zero)
    500.0,     # Payload
    15.0       # Speed
], dtype=np.float32)

predicted_action = get_action_from_onnx_model(dummy_raw_obs_for_inference)
print(f"\nPredicted action from ONNX model for dummy raw observation: {predicted_action}")

# You would integrate this `get_action_from_onnx_model` function into your drone's control loop.

Scaler loaded successfully.
ONNX model loaded. Input: obs, Output: action

Predicted action from ONNX model for dummy raw observation: [ 0.0073787  -0.09439701 -0.996511  ]
